In [1]:
import pandas as pd
%run Caminhos.ipynb

tabela_clientes = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_clientes_parquet}')
tabela_ind_estrategico = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_ind_estrategico_parquet}')
tabela_alunos = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_alunos_parquet}')
tabela_especificacao_ind_O_E = pd.read_parquet(f'{pasta_hierarquia}{caminho_especificacao_ind_O_E_parquet}')
tabela_base_gerada = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_base_gerada_parquet}')
tabela_professor = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_professor_parquet}')
tabela_notas = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_notas_parquet}')
tabela_notas_geral = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_notas_geral_parquet}')
tabela_frequencia_ativo = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_frequencia_ativo_parquet}')
tabela_frequencia_ITA = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_frequencia_ITA_parquet}')
tabela_Tempo_Integral = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_Tempo_Integral_parquet}')

85 - % de estudantes da rede estadual com frequência regular em 75%

In [5]:
tab_qtd_freq = tabela_frequencia_ativo

## Soma as frequencias dos estudantes inscritos nas turmas ITA/IME
freq_reg = tab_qtd_freq.merge(tabela_clientes[["idMatricula", "Territorio Desenvolv", "Matricula_2024", "Matricula_2025", "Matricula_2026"]], on='idMatricula', how='left')
freq_reg = freq_reg.drop_duplicates(subset=["idMatricula", "Territorio Desenvolv"])
freq_reg_aprov = freq_reg[freq_reg['percPresenca'] >= 75]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
freq_apr_24 = freq_reg_aprov[freq_reg_aprov['Matricula_2024'] == 1]
freq_apr_25 = freq_reg_aprov[freq_reg_aprov['Matricula_2025'] == 1]

soma_freq_apr_24 = freq_apr_24.groupby('Territorio Desenvolv')['idMatricula'].count().reset_index()
soma_freq_apr_25 = freq_apr_25.groupby('Territorio Desenvolv')['idMatricula'].count().reset_index()

# Adicionar coluna do ano
soma_freq_apr_24['Ano'] = 2024
soma_freq_apr_25['Ano'] = 2025

# Concatenar os DataFrames
soma_freq_apr = pd.concat([soma_freq_apr_24, soma_freq_apr_25])
#display(soma_freq_apr)
'----------------------------------------------------------------------------------------------------------------------------------------'
freq_reg_reprov = freq_reg[freq_reg['percPresenca'] < 75]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
freq_rep_24 = freq_reg_reprov[freq_reg_reprov['Matricula_2024'] == 1]
freq_rep_25 = freq_reg_reprov[freq_reg_reprov['Matricula_2025'] == 1]

soma_freq_rep_24 = freq_rep_24.groupby('Territorio Desenvolv')['idMatricula'].count().reset_index()
soma_freq_rep_25 = freq_rep_25.groupby('Territorio Desenvolv')['idMatricula'].count().reset_index()

# Adicionar coluna do ano
soma_freq_rep_24['Ano'] = 2024
soma_freq_rep_25['Ano'] = 2025

# Concatenar os DataFrames
soma_freq_rep = pd.concat([soma_freq_rep_24, soma_freq_rep_25])
#display(soma_freq_rep)
'----------------------------------------------------------------------------------------------------------------------------------------'
est_freq = pd.concat([soma_freq_apr, soma_freq_rep])
#display(est_freq)
'----------------------------------------------------------------------------------------------------------------------------------------'
# Matriculas totais
sum_est_freq = est_freq.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].sum().reset_index()
sum_est_freq = sum_est_freq.rename(columns={'idMatricula': 'Qtde_Total'})
#display(sum_est_freq)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
freq_aprovativa = soma_freq_apr.merge(sum_est_freq, on=['Territorio Desenvolv', 'Ano'])
freq_aprovativa['% de estudantes da rede estadual com frequência regular em 75%'] = freq_aprovativa['idMatricula'] / freq_aprovativa['Qtde_Total']
freq_aprovativa = freq_aprovativa[["Territorio Desenvolv", "Ano", "% de estudantes da rede estadual com frequência regular em 75%"]]
display(freq_aprovativa)

,Territorio Desenvolv,Ano,% de estudantes da rede estadual com frequência regular em 75%
0,CARAUBAIS,2024,0.936688
1,CHAPADA DAS MANGABEIRAS,2024,0.897593
2,CHAPADA VALE DO ITAIM,2024,0.875781
3,COCAIS,2024,0.948739
4,ENTRE RIOS,2024,0.875431
5,PLANICÍE LITORANEA,2024,0.906099
6,SERRA DA CAPIVARA,2024,0.917256
7,TABULEIROS DO ALTO PARNAÍBA,2024,0.904169
8,VALE DO CANINDÉ,2024,0.954848
9,VALE DO RIO GUARIBAS,2024,0.877541


385 - Indicador - % frequência dos estudantes inscritos nas turmas ITA/IME

In [6]:
tab_freq = tabela_frequencia_ITA

## Qtd aulas com presença dos estudantes inscritos nas turmas ITA/IME
tab_freq = tab_freq.merge(tabela_clientes[["idMatricula", "Territorio Desenvolv", "Matricula_2024", "Matricula_2025", "Matricula_2026"]], on='idMatricula', how='left')

# Filtrar o DataFrame onde tipoFrequencia = Presença
tab_freq_p = tab_freq[(tab_freq['tipoFrequencia'] == "Presença")]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
freq_24 = tab_freq_p[tab_freq_p['Matricula_2024'] == 1]
freq_25 = tab_freq_p[tab_freq_p['Matricula_2025'] == 1]

soma_freq_24 = freq_24.groupby(['Territorio Desenvolv', 'Ano'])['idAula'].count().reset_index()
soma_freq_25 = freq_25.groupby(['Territorio Desenvolv', 'Ano'])['idAula'].count().reset_index()

# Concatenar os DataFrames
soma_freq = pd.concat([soma_freq_24, soma_freq_25])
soma_freq = soma_freq.rename(columns={'idAula': 'Qtde aulas com presença'})
soma_freq['Ano'] = soma_freq['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente Linguas Estrangeiras
freq_24_total = tab_freq[tab_freq['Matricula_2024'] == 1]
freq_25_total = tab_freq[tab_freq['Matricula_2025'] == 1]

# Contar o número de idMatriculas distintas por Territorio Desenvolv
estudantes_freq_24  = freq_24_total.groupby(['Territorio Desenvolv', 'Ano'])['idAula'].count().reset_index()
estudantes_freq_25  = freq_25_total.groupby(['Territorio Desenvolv', 'Ano'])['idAula'].count().reset_index()

# Concatenar os DataFrames
estudantes_freq = pd.concat([estudantes_freq_24, estudantes_freq_25])
estudantes_freq = estudantes_freq.rename(columns={'idAula': 'Qtde aulas'})
estudantes_freq['Ano'] = estudantes_freq['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
base_nota_freq = soma_freq.merge(estudantes_freq, on=['Territorio Desenvolv', 'Ano'])
base_nota_freq['% frequência dos estudantes inscritos nas turmas ITA/IME'] = base_nota_freq['Qtde aulas com presença'] / base_nota_freq['Qtde aulas']
base_nota_freq = base_nota_freq[["Territorio Desenvolv", "Ano", "% frequência dos estudantes inscritos nas turmas ITA/IME"]]
display(base_nota_freq)

,Territorio Desenvolv,Ano,% frequência dos estudantes inscritos nas turmas ITA/IME
0,ENTRE RIOS,2024,0.999833


381 - Indicador - Média nas avaliações nas demais disciplinas

In [7]:
tab_nota_ITA = tabela_notas_geral[["Territorio Desenvolv", "idMatricula", "Ano", "idTurma", "nomeDisciplina", "nomeTipoNota", "valorNota", "idDisciplina", "Matricula_2024", "Matricula_2025", "Matricula_2026"]]

## Soma das notas dos estudantes da Rede no componente curricular

# Filtrar o DataFrame onde turma = ITA/IME
tab_nota_ITA = tab_nota_ITA[(tab_nota_ITA['idTurma'].isin([287816, 287818]))]
# Filtrar o DataFrame onde nomeDisciplina
tab_nota_ITA = tab_nota_ITA[(~tab_nota_ITA['idDisciplina'].isin([2551, 2556, 2557, 2558, 2559, 2560, 2561, 2562, 2563, 2564, 2565, 2566, 2567, 594, 2483, 2554, 595]))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
nota_ITA_24 = tab_nota_ITA[tab_nota_ITA['Matricula_2024'] == 1]
nota_ITA_25 = tab_nota_ITA[tab_nota_ITA['Matricula_2025'] == 1]

soma_nota_ITA_24 = nota_ITA_24.groupby(['Territorio Desenvolv', 'Ano'])['valorNota'].sum().reset_index()
soma_nota_ITA_25 = nota_ITA_25.groupby(['Territorio Desenvolv', 'Ano'])['valorNota'].sum().reset_index()

# Concatenar os DataFrames
soma_nota_ITA = pd.concat([soma_nota_ITA_24, soma_nota_ITA_25])
soma_nota_ITA['Ano'] = soma_nota_ITA['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente Linguas Estrangeiras

# Contar o número de idMatriculas distintas por Territorio Desenvolv
estudantes_ITA_24  = nota_ITA_24.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].count().reset_index()
estudantes_ITA_25  = nota_ITA_25.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].count().reset_index()

# Concatenar os DataFrames
estudantes_ITA = pd.concat([estudantes_ITA_24, estudantes_ITA_25])
estudantes_ITA['Ano'] = estudantes_ITA['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
base_nota_media_ITA = soma_nota_ITA.merge(estudantes_ITA, on=['Territorio Desenvolv', 'Ano'])
base_nota_media_ITA['Média nas avaliações nas demais disciplinas'] = base_nota_media_ITA['valorNota'] / base_nota_media_ITA['idMatricula']

base_nota_media_ITA_IME = base_nota_media_ITA.drop(['valorNota', 'idMatricula'], axis=1)
#base_nota_media_ITA_IME = base_nota_media_ITA_IME[(base_nota_media_ITA_IME["Territorio Desenvolv"] == 3)]
display(base_nota_media_ITA_IME)

,Territorio Desenvolv,Ano,Média nas avaliações nas demais disciplinas
0,ENTRE RIOS,2024,7.554592


382 - Indicador - Média nas avaliações das disciplinas especificas ITA/IME

In [8]:
tab_nota_esp_ITA = tabela_notas_geral[["Territorio Desenvolv", "idMatricula", "idTurma", "Ano", "nomeDisciplina", "nomeTipoNota", "valorNota", "idDisciplina", "Matricula_2024", "Matricula_2025", "Matricula_2026"]]

## Soma das notas dos estudantes da Rede no componente curricular
# Filtrar o DataFrame onde turma = ITA/IME
tab_nota_esp_ITA = tab_nota_esp_ITA[(tab_nota_esp_ITA['idTurma'].isin([287816, 287818]))]
# Filtrar o DataFrame onde nomeDisciplina
tab_nota_esp_ITA = tab_nota_esp_ITA[(tab_nota_esp_ITA['idDisciplina'].isin([2551, 2556, 2557, 2558, 2559, 2560, 2561, 2562, 2563, 2564, 2565, 2566, 2567, 594, 2483, 2554, 595]))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
nota_esp_ITA_24 = tab_nota_esp_ITA[tab_nota_esp_ITA['Matricula_2024'] == 1]
nota_esp_ITA_25 = tab_nota_esp_ITA[tab_nota_esp_ITA['Matricula_2025'] == 1]

soma_nota_esp_ITA_24 = nota_esp_ITA_24.groupby(['Territorio Desenvolv', 'Ano'])['valorNota'].sum().reset_index()
soma_nota_esp_ITA_25 = nota_esp_ITA_25.groupby(['Territorio Desenvolv', 'Ano'])['valorNota'].sum().reset_index()

# Concatenar os DataFrames
soma_nota_esp_ITA = pd.concat([soma_nota_esp_ITA_24, soma_nota_esp_ITA_25])
soma_nota_esp_ITA['Ano'] = soma_nota_esp_ITA['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente Linguas Estrangeiras

# Contar o número de idMatriculas distintas por Territorio Desenvolv
estudantes_esp_ITA_24  = nota_esp_ITA_24.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].count().reset_index()
estudantes_esp_ITA_25  = nota_esp_ITA_25.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].count().reset_index()

# Concatenar os DataFrames
estudantes_esp_ITA = pd.concat([estudantes_esp_ITA_24, estudantes_esp_ITA_25])
estudantes_esp_ITA['Ano'] = estudantes_esp_ITA['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
base_nota_media_esp_ITA = soma_nota_esp_ITA.merge(estudantes_esp_ITA, on=['Territorio Desenvolv', 'Ano'])
base_nota_media_esp_ITA['Média nas avaliações das disciplinas especificas ITA/IME'] = base_nota_media_esp_ITA['valorNota'] / base_nota_media_esp_ITA['idMatricula']

base_nota_media_esp_ITA_IME = base_nota_media_esp_ITA.drop(['valorNota', 'idMatricula'], axis=1)
#base_nota_media_esp_ITA_IME = base_nota_media_esp_ITA_IME[(base_nota_media_esp_ITA_IME["Territorio Desenvolv"] == 3)]
display(base_nota_media_esp_ITA_IME)

,Territorio Desenvolv,Ano,Média nas avaliações das disciplinas especificas ITA/IME
0,ENTRE RIOS,2024,7.319304


402 - Indicador - Média dos alunos da rede nas disciplinas de línguas estrangeiras

In [9]:
tab_nota_LEst = tabela_notas_geral[["Territorio Desenvolv", "idMatricula", "Ano", "nomeDisciplina", "nomeTipoNota", "valorNota", "idDisciplina", "Matricula_2024", "Matricula_2025", "Matricula_2026"]]

## Soma das notas dos estudantes da Rede no componente curricular Linguas Estrangeiras

# Filtrar o DataFrame onde nomeDisciplina = LÍNGUAS ESTRANGEIRAS
nota_m_LEst = tab_nota_LEst[(tab_nota_LEst['idDisciplina'].isin([2400, 2233, 2232, 11, 10, 9]))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
nota_LEst_24 = nota_m_LEst[nota_m_LEst['Matricula_2024'] == 1]
nota_LEst_25 = nota_m_LEst[nota_m_LEst['Matricula_2025'] == 1]

soma_nota_LEst_24 = nota_LEst_24.groupby(['Territorio Desenvolv', 'Ano'])['valorNota'].sum().reset_index()
soma_nota_LEst_25 = nota_LEst_25.groupby(['Territorio Desenvolv', 'Ano'])['valorNota'].sum().reset_index()

# Concatenar os DataFrames
soma_nota_LEst = pd.concat([soma_nota_LEst_24, soma_nota_LEst_25])
soma_nota_LEst['Ano'] = soma_nota_LEst['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente Linguas Estrangeiras

# Contar o número de idMatriculas distintas por Territorio Desenvolv
estudantes_LEst_24  = nota_LEst_24.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].count().reset_index()
estudantes_LEst_25  = nota_LEst_25.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].count().reset_index()

# Concatenar os DataFrames
estudantes_LEst = pd.concat([estudantes_LEst_24, estudantes_LEst_25])
soma_nota_LEst['Ano'] = soma_nota_LEst['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
base_nota_media_LEst = soma_nota_LEst.merge(estudantes_LEst, on=['Territorio Desenvolv', 'Ano'])
base_nota_media_LEst['Média dos alunos da rede nas disciplinas de línguas estrangeiras'] = base_nota_media_LEst['valorNota'] / base_nota_media_LEst['idMatricula']

base_nota_media_LP = base_nota_media_LEst.drop(['valorNota', 'idMatricula'], axis=1)
#base_nota_media_LP = base_nota_media_LP[(base_nota_media_LP["Territorio Desenvolv"] == 3)]
display(base_nota_media_LP)

,Territorio Desenvolv,Ano,Média dos alunos da rede nas disciplinas de línguas estrangeiras
0,CARAUBAIS,2024,6.947283
1,CHAPADA DAS MANGABEIRAS,2024,6.588229
2,CHAPADA VALE DO ITAIM,2024,6.725232
3,COCAIS,2024,7.052176
4,ENTRE RIOS,2024,6.885377
5,PLANICÍE LITORANEA,2024,6.959216
6,SERRA DA CAPIVARA,2024,7.017089
7,TABULEIROS DO ALTO PARNAÍBA,2024,6.881314
8,VALE DO CANINDÉ,2024,6.944012
9,VALE DO RIO GUARIBAS,2024,6.852419


163 - Indicador - % de aprovação dos alunos do Ensino Médio - ok - A Validar

In [10]:
tab_nota_EM_Aprov_geral = tabela_notas_geral[["Territorio Desenvolv", "idMatricula", "Ano", "nomeTipoNota", "valorNota", "Matricula_2024", "Matricula_2025"]]
tab_client_EM_Aprov = tabela_clientes[["idMatricula", "Territorio Desenvolv", "Ano", "nivelEnsino", "EtapaAgregada", "grupo", "STATUS_INTEGRAL"]]

# Mesclar os DataFrames mantendo apenas as colunas adicionais da tab_client_LP_Aprovados
tab_nota_EM_Aprov_geral = tab_nota_EM_Aprov_geral.merge(tab_client_EM_Aprov, on=['idMatricula', 'Territorio Desenvolv', 'Ano'], how='inner')

#Filtro 'Ensino Médio' e nomeDisciplina = LÍNGUA PORTUGUESA
filt_EM = tab_nota_EM_Aprov_geral[
    (tab_nota_EM_Aprov_geral['nivelEnsino'].str.contains('Ensino Médio', regex=False)) &
    (tab_nota_EM_Aprov_geral['EtapaAgregada'].str.contains('Ensino Médio|Integrada', regex=True)) &
    (tab_nota_EM_Aprov_geral['grupo'].str.contains('Ensino', regex=False))
    ]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filt_EM_24 = filt_EM[filt_EM['Matricula_2024'] == 1]
filt_EM_25 = filt_EM[filt_EM['Matricula_2025'] == 1]

# Agrupando por "Id Escola" e "idMatricula" e calculando a média de "valorNota"
agrupar_media_24 = filt_EM_24.groupby(["Territorio Desenvolv", "Ano", "idMatricula", "nomeTipoNota"])["valorNota"].mean().reset_index()
agrupar_media_25 = filt_EM_25.groupby(["Territorio Desenvolv", "Ano", "idMatricula", "nomeTipoNota"])["valorNota"].mean().reset_index()

def contar_distintos(df):
    return df.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [agrupar_media_25, agrupar_media_24]
mat_EM_geral = pd.concat([contar_distintos(df) for df in dataframes])

mat_EM_geral = mat_EM_geral.rename(columns={'idMatricula': 'Qtde_Matriculas'})
"-----------------------------------------------------------------------------------------------------------------"
# Filtrar o DataFrame onde agregada = Ensino Médio
filt_EM_aprov_geral_24 = agrupar_media_24[((agrupar_media_24['valorNota'] >= 6))]
filt_EM_aprov_geral_25 = agrupar_media_25[((agrupar_media_25['valorNota'] >= 6))]

def contar_distintos(df):
    return df.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_EM_aprov_geral_25, filt_EM_aprov_geral_24]
aprovados_EM_geral = pd.concat([contar_distintos(df) for df in dataframes])

aprovados_EM_geral = aprovados_EM_geral.rename(columns={'idMatricula': 'Qtde_Matriculas_Aprovadas'})
"-----------------------------------------------------------------------------------------------------------------"
# Mesclar os dois DataFrames pelo "Id Escola"
EM_geral = mat_EM_geral.merge(aprovados_EM_geral, on=['Territorio Desenvolv', 'Ano'])
EM_geral['Ano'] = EM_geral['Ano'].astype(int)
"-----------------------------------------------------------------------------------------------------------------"
# % de aprovação dos alunos do Ensino Médio
EM_geral['% de aprovação dos alunos do Ensino Médio'] = EM_geral['Qtde_Matriculas_Aprovadas'] / EM_geral['Qtde_Matriculas']
#EM_geral = mat_EM_geral[mat_EM_geral["Id Escola"] == 3]
EM_geral = EM_geral[['Territorio Desenvolv', 'Ano', '% de aprovação dos alunos do Ensino Médio']]
display(EM_geral)

,Territorio Desenvolv,Ano,% de aprovação dos alunos do Ensino Médio
0,CARAUBAIS,2024,0.919217
1,CHAPADA DAS MANGABEIRAS,2024,0.891067
2,CHAPADA VALE DO ITAIM,2024,0.887345
3,COCAIS,2024,0.939542
4,ENTRE RIOS,2024,0.894278
5,PLANICÍE LITORANEA,2024,0.904584
6,SERRA DA CAPIVARA,2024,0.920626
7,TABULEIROS DO ALTO PARNAÍBA,2024,0.897799
8,VALE DO CANINDÉ,2024,0.851677
9,VALE DO RIO GUARIBAS,2024,0.897990


1193 - Indicador - % de estudantes do E.M. com média >= 6 no componente curricular Matemática - ok - Validado

In [11]:
tab_nota_M_Aprovados = tabela_notas[["Territorio Desenvolv", "idMatricula", "Ano", "mediaNota", "nomeDisciplina", "Matricula_2024", "Matricula_2025"]]
tab_client_M_Aprovados = tabela_clientes[["idMatricula", "Territorio Desenvolv", "Ano", "nivelEnsino", "EtapaAgregada", "grupo", "STATUS_INTEGRAL"]]

# Mesclar os DataFrames mantendo apenas as colunas adicionais da tab_client_LP_Aprovados
tab_nota_M_Aprovados = tab_nota_M_Aprovados.merge(tab_client_M_Aprovados, on=['idMatricula', 'Territorio Desenvolv', 'Ano'], how='inner')

#Filtro 'Ensino Médio' e nomeDisciplina = MATEMÁTICA
filt_M = tab_nota_M_Aprovados[
    (tab_nota_M_Aprovados['nomeDisciplina'] == "MATEMÁTICA") & 
    (tab_nota_M_Aprovados['nivelEnsino'].str.contains('Ensino Médio', regex=False)) &
    (tab_nota_M_Aprovados['EtapaAgregada'].str.contains('Ensino Médio|Integrada', regex=True)) &
    (tab_nota_M_Aprovados['grupo'].str.contains('Ensino', regex=False))
    ]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filt_M_24 = filt_M[filt_M['Matricula_2024'] == 1]
filt_M_25 = filt_M[filt_M['Matricula_2025'] == 1]

def contar_distintos(df):
    return df.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_M_25, filt_M_24]
soma_nota_M = pd.concat([contar_distintos(df) for df in dataframes])
"-----------------------------------------------------------------------------------------------------------------"
# Filtrar o DataFrame onde agregada = Ensino Médio
filt_M_aprovados_24 = filt_M_24[((filt_M_24['mediaNota'] >= 6))]
filt_M_aprovados_25 = filt_M_25[((filt_M_25['mediaNota'] >= 6))]

def contar_distintos(df):
    return df.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_M_aprovados_25, filt_M_aprovados_24]
aprovados_M = pd.concat([contar_distintos(df) for df in dataframes])

# Renomear a coluna para identificação clara
aprovados_M = aprovados_M.rename(columns={'idMatricula': 'Qtde_Matriculas_Aprovadas'})
"-----------------------------------------------------------------------------------------------------------------"
# Mesclar os dois DataFrames pelo "Territorio Desenvolv"
mat_M = soma_nota_M.merge(aprovados_M, on=['Territorio Desenvolv', 'Ano'])
mat_M['Ano'] = mat_M['Ano'].astype(int)

# Substituir valores NaN por 0 na nova coluna (caso uma escola não tenha nenhuma matrícula com nota >= 6)
mat_M['Qtde_Matriculas_Aprovadas'] = mat_M['Qtde_Matriculas_Aprovadas'].fillna(0)
"-----------------------------------------------------------------------------------------------------------------"
# % de estudantes do E.M. com média >= 6 no componente curricular Matemática
mat_M['% de estudantes do E.M. com média >= 6 no componente curricular Matemática'] = mat_M['Qtde_Matriculas_Aprovadas'] / mat_M['idMatricula']
mat_M = mat_M[["Territorio Desenvolv", "Ano", "% de estudantes do E.M. com média >= 6 no componente curricular Matemática"]]
display(mat_M)

,Territorio Desenvolv,Ano,% de estudantes do E.M. com média >= 6 no componente curricular Matemática
0,CARAUBAIS,2024,0.816102
1,CHAPADA DAS MANGABEIRAS,2024,0.722770
2,CHAPADA VALE DO ITAIM,2024,0.699190
3,COCAIS,2024,0.826817
4,ENTRE RIOS,2024,0.804778
5,PLANICÍE LITORANEA,2024,0.743132
6,SERRA DA CAPIVARA,2024,0.805080
7,TABULEIROS DO ALTO PARNAÍBA,2024,0.748913
8,VALE DO CANINDÉ,2024,0.793126
9,VALE DO RIO GUARIBAS,2024,0.738016


1194 - Indicador - % de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa - ok - Validado

In [12]:
tab_nota_LP_Aprovados = tabela_notas[["Territorio Desenvolv", "idMatricula", "Ano", "mediaNota", "nomeDisciplina", "Matricula_2024", "Matricula_2025"]]
tab_client_LP_Aprovados = tabela_clientes[["idMatricula", "Territorio Desenvolv", "Ano", "nivelEnsino", "EtapaAgregada", "grupo", "STATUS_INTEGRAL"]]

# Mesclar os DataFrames mantendo apenas as colunas adicionais da tab_client_LP_Aprovados
tab_nota_LP_Aprovados = tab_nota_LP_Aprovados.merge(tab_client_LP_Aprovados, on=['idMatricula', 'Territorio Desenvolv', 'Ano'], how='inner')

#Filtro 'Ensino Médio' e nomeDisciplina = LÍNGUA PORTUGUESA
filt_LP = tab_nota_LP_Aprovados[
    (tab_nota_LP_Aprovados['nomeDisciplina'] == "LÍNGUA PORTUGUESA") & 
    (tab_nota_LP_Aprovados['nivelEnsino'].str.contains('Ensino Médio', regex=False)) &
    (tab_nota_LP_Aprovados['EtapaAgregada'].str.contains('Ensino Médio|Integrada', regex=True)) &
    (tab_nota_LP_Aprovados['grupo'].str.contains('Ensino', regex=False))
    ]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filt_LP_24 = filt_LP[filt_LP['Matricula_2024'] == 1]
filt_LP_25 = filt_LP[filt_LP['Matricula_2025'] == 1]

def contar_distintos(df):
    return df.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_LP_25, filt_LP_24]
soma_nota_LP = pd.concat([contar_distintos(df) for df in dataframes])
"-----------------------------------------------------------------------------------------------------------------"
# Filtrar o DataFrame onde agregada = Ensino Médio
filt_LP_aprovados_24 = filt_LP_24[((filt_LP_24['mediaNota'] >= 6))]
filt_LP_aprovados_25 = filt_LP_25[((filt_LP_25['mediaNota'] >= 6))]

def contar_distintos(df):
    return df.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_LP_aprovados_25, filt_LP_aprovados_24]
aprovados_nota_LP = pd.concat([contar_distintos(df) for df in dataframes])

aprovados_nota_LP = aprovados_nota_LP.rename(columns={'idMatricula': 'Qtde_Matriculas_Aprovadas'})
"-----------------------------------------------------------------------------------------------------------------"
# Mesclar os dois DataFrames pelo "Territorio Desenvolv"
nota_LP = soma_nota_LP.merge(aprovados_nota_LP, on=['Territorio Desenvolv', 'Ano'])
nota_LP['Ano'] = nota_LP['Ano'].astype(int)
"-----------------------------------------------------------------------------------------------------------------"
# % de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa
nota_LP['% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa'] = nota_LP['Qtde_Matriculas_Aprovadas'] / nota_LP['idMatricula']
#mat_LP = mat_LP[['Territorio Desenvolv', '% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa']]
nota_LP = nota_LP[["Territorio Desenvolv", "Ano", "% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa"]]
display(nota_LP)

,Territorio Desenvolv,Ano,% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa
0,CARAUBAIS,2024,0.795780
1,CHAPADA DAS MANGABEIRAS,2024,0.812337
2,CHAPADA VALE DO ITAIM,2024,0.809428
3,COCAIS,2024,0.887550
4,ENTRE RIOS,2024,0.862887
5,PLANICÍE LITORANEA,2024,0.818182
6,SERRA DA CAPIVARA,2024,0.854257
7,TABULEIROS DO ALTO PARNAÍBA,2024,0.778671
8,VALE DO CANINDÉ,2024,0.730441
9,VALE DO RIO GUARIBAS,2024,0.859826


2237 - Indicador - Nota média dos estudantes da Rede no componente curricular Língua Portuguesa - ok

In [13]:
tab_nota_LP = tabela_notas_geral[["Territorio Desenvolv", "idMatricula", "Ano", "idDisciplina", "agregada", "nomeTipoNota", "valorNota", "Matricula_2024", "Matricula_2025", "Matricula_2026"]]

## Soma das notas dos estudantes da Rede no componente curricular LP

# Filtrar o DataFrame onde nomeDisciplina = LÍNGUA PORTUGUESA
nota_m_LP = tab_nota_LP[(tab_nota_LP['idDisciplina'] == 1)]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
nota_LP_24 = nota_m_LP[nota_m_LP['Matricula_2024'] == 1]
nota_LP_25 = nota_m_LP[nota_m_LP['Matricula_2025'] == 1]

soma_nota_LP_24 = nota_LP_24.groupby(['Territorio Desenvolv', 'Ano'])['valorNota'].sum().reset_index()
soma_nota_LP_25 = nota_LP_25.groupby(['Territorio Desenvolv', 'Ano'])['valorNota'].sum().reset_index()

# Concatenar os DataFrames
soma_nota_LP = pd.concat([soma_nota_LP_24, soma_nota_LP_25])
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente LP

# Contar o número de idMatriculas distintas por Territorio Desenvolv
estudantes_LP_24  = nota_LP_24.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].count().reset_index()
estudantes_LP_25  = nota_LP_25.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].count().reset_index()

# Concatenar os DataFrames
estudantes_LP = pd.concat([estudantes_LP_24, estudantes_LP_25])
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
nota_media_LP = soma_nota_LP.merge(estudantes_LP, on=['Territorio Desenvolv', 'Ano'])
nota_media_LP['Ano'] = nota_media_LP['Ano'].astype(int)
nota_media_LP['Nota média dos estudantes da Rede no componente curricular Língua Portuguesa'] = nota_media_LP['valorNota'] / nota_media_LP['idMatricula']

nota_media_LP = nota_media_LP.drop(['valorNota', 'idMatricula'], axis=1)
#nota_media_LP = nota_media_LP[(nota_media_LP["Territorio Desenvolv"] == 3)]
display(nota_media_LP)

,Territorio Desenvolv,Ano,Nota média dos estudantes da Rede no componente curricular Língua Portuguesa
0,CARAUBAIS,2024,6.402021
1,CHAPADA DAS MANGABEIRAS,2024,6.345411
2,CHAPADA VALE DO ITAIM,2024,6.264034
3,COCAIS,2024,6.741004
4,ENTRE RIOS,2024,6.634906
5,PLANICÍE LITORANEA,2024,6.685486
6,SERRA DA CAPIVARA,2024,6.612097
7,TABULEIROS DO ALTO PARNAÍBA,2024,6.477340
8,VALE DO CANINDÉ,2024,6.222108
9,VALE DO RIO GUARIBAS,2024,6.588612


2238 - Indicador - Nota média dos estudantes da Rede no componente curricular Matemática - OK

In [14]:
tab_nota_M = tabela_notas_geral[["Territorio Desenvolv", "idMatricula", "Ano", "idDisciplina", "agregada", "nomeTipoNota", "valorNota", "Matricula_2024", "Matricula_2025", "Matricula_2026"]]

## Soma das notas dos estudantes da Rede no componente curricular M

# Filtrar o DataFrame onde nomeDisciplina = LÍNGUA PORTUGUESA
nota_M = tab_nota_M[(tab_nota_M['idDisciplina'] == 5)]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
nota_M_24 = nota_M[nota_M['Matricula_2024'] == 1]
nota_M_25 = nota_M[nota_M['Matricula_2025'] == 1]

soma_nota_M_24 = nota_M_24.groupby(['Territorio Desenvolv', 'Ano'])['valorNota'].sum().reset_index()
soma_nota_M_25 = nota_M_25.groupby(['Territorio Desenvolv', 'Ano'])['valorNota'].sum().reset_index()

# Concatenar os DataFrames
soma_nota_M = pd.concat([soma_nota_M_24, soma_nota_M_25])
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente M

# Contar o número de idMatriculas distintas por Id Escola
estudantes_M_24 = nota_M_24.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].count().reset_index()
estudantes_M_25 = nota_M_25.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].count().reset_index()

# Concatenar os DataFrames
estudantes_M = pd.concat([estudantes_M_24, estudantes_M_25])
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
base_nota_media_M = soma_nota_M.merge(estudantes_M, on=['Territorio Desenvolv', 'Ano'])
base_nota_media_M['Ano'] = base_nota_media_M['Ano'].astype(int)

base_nota_media_M['Nota média dos estudantes da Rede no componente curricular Matemática'] = base_nota_media_M['valorNota'] / base_nota_media_M['idMatricula']
base_nota_media_M = base_nota_media_M.drop(['valorNota', 'idMatricula'], axis=1)
display(base_nota_media_M)

,Territorio Desenvolv,Ano,Nota média dos estudantes da Rede no componente curricular Matemática
0,CARAUBAIS,2024,6.364896
1,CHAPADA DAS MANGABEIRAS,2024,5.908017
2,CHAPADA VALE DO ITAIM,2024,5.998682
3,COCAIS,2024,6.434407
4,ENTRE RIOS,2024,6.261973
5,PLANICÍE LITORANEA,2024,6.347022
6,SERRA DA CAPIVARA,2024,6.278368
7,TABULEIROS DO ALTO PARNAÍBA,2024,6.158774
8,VALE DO CANINDÉ,2024,6.227370
9,VALE DO RIO GUARIBAS,2024,6.204463


2227 - Indicador - Nº de docentes autodeclarados pretos/pardos na rede estadual

2225 - Indicador - Nº de docentes autodeclarados indígenas na rede estadual

2215 - Indicador - N° de estudantes autodeclarados pretos matriculados na rede - ok

In [15]:
tabela_alunos_est = tabela_alunos[["idAluno", "codigoEscolarAluno", "RA", "nome", "sexo", "raca"]]

# Filtrar alunos com raca 'Preta'
tabela_alunos_preta = tabela_alunos_est[tabela_alunos_est['raca'] == 'Preta']

# Mesclar os DataFrames para adicionar as colunas com o cálculo
Matricula_Alunos = tabela_alunos_preta.merge(tabela_clientes[["idAluno", "idMatricula", "Territorio Desenvolv", "Matricula_2026", "Matricula_2025", "Matricula_2024", "Ano"]], on=['idAluno'], how='inner')

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filter_df_24 = Matricula_Alunos[Matricula_Alunos['Matricula_2024'] == 1]
filter_df_25 = Matricula_Alunos[Matricula_Alunos['Matricula_2025'] == 1]

# Contar os valores distintos na coluna idMatricula por municipio (idMunicipio)
distinct_count_Preto_24_mun = filter_df_24.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()
distinct_count_Preto_25_mun = filter_df_25.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames
matricula_aut_pretos = pd.concat([distinct_count_Preto_24_mun, distinct_count_Preto_25_mun])
matricula_aut_pretos.rename(columns={'idMatricula': 'N° de estudantes autodeclarados pretos matriculados na rede'}, inplace=True)
matricula_aut_pretos['Ano'] = matricula_aut_pretos['Ano'].astype(int)
display(matricula_aut_pretos)

,Territorio Desenvolv,Ano,N° de estudantes autodeclarados pretos matriculados na rede
0,CARAUBAIS,2024,211
1,CHAPADA DAS MANGABEIRAS,2024,478
2,CHAPADA VALE DO ITAIM,2024,755
3,COCAIS,2024,570
4,ENTRE RIOS,2024,4378
5,PLANICÍE LITORANEA,2024,516
6,SERRA DA CAPIVARA,2024,379
7,TABULEIROS DO ALTO PARNAÍBA,2024,740
8,VALE DO CANINDÉ,2024,142
9,VALE DO RIO GUARIBAS,2024,342


2196 - Indicador - N° de estudantes autodeclarados indígenas matriculados na rede - ok - A Validar

In [16]:
# Filtrar alunos com raca 'Preta'
tabela_alunos_Ind = tabela_alunos_est[tabela_alunos_est['raca'] == 'Indígena']

num_linhas = tabela_alunos_est.shape[0]

# Mesclar os DataFrames para adicionar as colunas com o cálculo
Matricula_Alunos_Ind = tabela_alunos_Ind.merge(tabela_clientes[["idAluno", "idMatricula", "Territorio Desenvolv", "Matricula_2026", "Matricula_2025", "Matricula_2024", "Ano"]], on=['idAluno'], how='inner')

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filter_Ind_df_24 = Matricula_Alunos_Ind[Matricula_Alunos_Ind['Matricula_2024'] == 1]
filter_Ind_df_25 = Matricula_Alunos_Ind[Matricula_Alunos_Ind['Matricula_2025'] == 1]

# Contar os valores distintos na coluna idMatricula por escola (Id Escola)
idMatricula_Ind_24_escola = filter_Ind_df_24.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()
idMatricula_Ind_25_escola = filter_Ind_df_25.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames
matricula_aut_indigena = pd.concat([idMatricula_Ind_24_escola, idMatricula_Ind_25_escola])
matricula_aut_indigena.rename(columns={'idMatricula': 'N° de estudantes autodeclarados indígenas matriculados na rede'}, inplace=True)
matricula_aut_indigena['Ano'] = matricula_aut_indigena['Ano'].astype(int)
display(matricula_aut_indigena)

,Territorio Desenvolv,Ano,N° de estudantes autodeclarados indígenas matriculados na rede
0,CARAUBAIS,2024,10
1,CHAPADA VALE DO ITAIM,2024,5
2,COCAIS,2024,49
3,ENTRE RIOS,2024,141
4,SERRA DA CAPIVARA,2024,1
5,TABULEIROS DO ALTO PARNAÍBA,2024,1
6,VALE DO CANINDÉ,2024,13
7,VALE DO RIO GUARIBAS,2024,1
8,VALE DOS RIOS PIAUÍ E ITAUEIRA,2024,14


153 - Indicador - % de municípios com escolas da rede estadual em tempo integral

In [17]:
# Qtde de municipio com escolas da rede estadual
tabela_clientes_matricula_est = tabela_clientes[["idMatricula", "Territorio Desenvolv", "Ano", "Id Municipio", "Matricula_2023", "Matricula_2024", "Matricula_2025"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
mun_24 = tabela_clientes_matricula_est[tabela_clientes_matricula_est['Matricula_2024'] == 1]
mun_25 = tabela_clientes_matricula_est[tabela_clientes_matricula_est['Matricula_2025'] == 1]

# Contar os valores distintos na coluna idMatricula por escola (Territorio Desenvolv)
mun_24_per_mun = mun_24.groupby(['Territorio Desenvolv', 'Ano'])['Id Municipio'].nunique().reset_index()
mun_25_per_mun = mun_25.groupby(['Territorio Desenvolv', 'Ano'])['Id Municipio'].nunique().reset_index()

# Concatenar os DataFrames
combined_mun = pd.concat([mun_24_per_mun, mun_25_per_mun])

combined_mun.rename(columns={'Id Municipio': 'Qtde de municipio com escolas da rede estadual'}, inplace=True)
combined_mun['Ano'] = combined_mun['Ano'].astype(int)
display(combined_mun)

,Territorio Desenvolv,Ano,Qtde de municipio com escolas da rede estadual
0,CARAUBAIS,2024,16
1,CHAPADA DAS MANGABEIRAS,2024,18
2,CHAPADA VALE DO ITAIM,2024,23
3,COCAIS,2024,22
4,ENTRE RIOS,2024,31
5,PLANICÍE LITORANEA,2024,11
6,SERRA DA CAPIVARA,2024,26
7,TABULEIROS DO ALTO PARNAÍBA,2024,14
8,VALE DO CANINDÉ,2024,5
9,VALE DO RIO GUARIBAS,2024,14


In [18]:
# Quantidade de municipio com escolas da rede estadual em tempo integral
tabela_mun_integral = tabela_Tempo_Integral[["Territorio Desenvolv", "Id Escola", "Id Municipio", "Ano", "STATUS_INTEGRAL"]]

filt_status_mun = tabela_mun_integral[tabela_mun_integral['STATUS_INTEGRAL'] == 1]

# FIltrar o Dataframe onde Matricula_2024 e 2023 é igual a 1
filt_24_mun = filt_status_mun[filt_status_mun['Ano'] == 2024]
filt_25_mun = filt_status_mun[filt_status_mun['Ano'] == 2025]

# Somar os valores na coluna Status_integral por matricula (Id Municipio)
sum_24_mun = filt_24_mun.groupby(['Territorio Desenvolv', 'Ano'])['Id Municipio'].nunique().reset_index()
sum_25_mun = filt_25_mun.groupby(['Territorio Desenvolv', 'Ano'])['Id Municipio'].nunique().reset_index()

# Concatenar os Dataframes
comb_int_mun = pd.concat([sum_24_mun, sum_25_mun])
comb_int_mun.rename(columns={'Id Municipio': 'Qtde de Municipio com escolas de tempo integral'}, inplace=True)
"----------------------------------------------------------------------------------------------------------------"
# Mesclagem para DF final
resultado_mun = combined_mun.merge(comb_int_mun, on=['Territorio Desenvolv', 'Ano'], how='left')
"----------------------------------------------------------------------------------------------------------------"
# Criar uma nova coluna com a divisão das colunas 'Nº Matriculas Total (EPT)' e 'Nº total de matrículas da rede estadual de educação'
resultado_mun['% de municípios com escolas da rede estadual em tempo integral'] = resultado_mun['Qtde de Municipio com escolas de tempo integral'] / resultado_mun['Qtde de municipio com escolas da rede estadual']
resultado_mun = resultado_mun[["Territorio Desenvolv", "% de municípios com escolas da rede estadual em tempo integral", "Ano"]]
display(resultado_mun)

,Territorio Desenvolv,% de municípios com escolas da rede estadual em tempo integral,Ano
0,CARAUBAIS,0.250000,2024
1,CHAPADA DAS MANGABEIRAS,0.500000,2024
2,CHAPADA VALE DO ITAIM,0.652174,2024
3,COCAIS,0.363636,2024
4,ENTRE RIOS,0.612903,2024
5,PLANICÍE LITORANEA,0.272727,2024
6,SERRA DA CAPIVARA,0.423077,2024
7,TABULEIROS DO ALTO PARNAÍBA,0.357143,2024
8,VALE DO CANINDÉ,0.600000,2024
9,VALE DO RIO GUARIBAS,0.714286,2024


159 - Indicador - Nº total de matrículas da rede estadual de educação

In [19]:
# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filtered_df_24 = tabela_clientes_matricula_est[tabela_clientes_matricula_est['Matricula_2024'] == 1]
filtered_df_23 = tabela_clientes_matricula_est[tabela_clientes_matricula_est['Matricula_2023'] == 1]

def contar_distintos(df):
    return df.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filtered_df_23, filtered_df_24]
combined_df = pd.concat([contar_distintos(df) for df in dataframes])

combined_df.rename(columns={'idMatricula': 'Nº total de matrículas da rede estadual de educação'}, inplace=True)
combined_df['Ano'] = combined_df['Ano'].astype(int)
display(combined_df)

,Territorio Desenvolv,Ano,Nº total de matrículas da rede estadual de educação
0,CARAUBAIS,2023,14016
1,CHAPADA DAS MANGABEIRAS,2023,9356
2,CHAPADA VALE DO ITAIM,2023,14522
3,COCAIS,2023,23687
4,ENTRE RIOS,2023,66722
5,PLANICÍE LITORANEA,2023,15198
6,SERRA DA CAPIVARA,2023,16223
7,TABULEIROS DO ALTO PARNAÍBA,2023,6906
8,VALE DO CANINDÉ,2023,3283
9,VALE DO RIO GUARIBAS,2023,7017


2202 - Indicador - % de matrículas de tempo integral no ensino regular, na rede estadual

In [20]:
# Quantidade de matricula de tempo integral por estado
tabela_integral = tabela_Tempo_Integral[["idMatricula", "Territorio Desenvolv", "Ano", "tempoIntegral"]]
tab_matr = tabela_clientes[["idMatricula", "Territorio Desenvolv", "Ano", "Id Municipio", "Matricula_2024", "Matricula_2025"]]

# Mesclar os DataFrames mantendo apenas as colunas adicionais da tab_client_LP_Aprovados
tabela_integral = tabela_integral.merge(tab_matr, on=['idMatricula', 'Territorio Desenvolv', 'Ano'], how='inner')
filt_status = tabela_integral[tabela_integral['tempoIntegral'] == 1]

# FIltrar o Dataframe onde Matricula_2024 e 2023 é igual a 1
filt_24 = filt_status[filt_status['Matricula_2024'] == 1]
filt_23 = filt_status[filt_status['Matricula_2025'] == 1]

def contar_distintos(df):
    return df.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_23, filt_24]
comb_int = pd.concat([contar_distintos(df) for df in dataframes])

comb_int.rename(columns={'idMatricula': 'Qtde de matricula de tempo integral'}, inplace=True)
display(comb_int)
"----------------------------------------------------------------------------------------------------------------"
# Mesclagem para DF final
resultado_final = combined_df.merge(comb_int, on=['Territorio Desenvolv', 'Ano'], how='inner')
resultado_final['Ano'] = resultado_final['Ano'].astype(int)
"----------------------------------------------------------------------------------------------------------------"
# Criar uma nova coluna com a divisão das colunas 'Nº Matriculas Total (EPT)' e 'Nº total de matrículas da rede estadual de educação'
resultado_final['% de matrículas de tempo integral no ensino regular, na rede estadual'] = resultado_final['Qtde de matricula de tempo integral'] / resultado_final['Nº total de matrículas da rede estadual de educação']
resultado_final = resultado_final[["Territorio Desenvolv", "% de matrículas de tempo integral no ensino regular, na rede estadual", "Ano"]]
display(resultado_final)


,Territorio Desenvolv,Ano,Qtde de matricula de tempo integral
0,CARAUBAIS,2024.0,1684
1,CHAPADA DAS MANGABEIRAS,2024.0,2166
2,CHAPADA VALE DO ITAIM,2024.0,3597
3,COCAIS,2024.0,4420
4,ENTRE RIOS,2024.0,14349
5,PLANICÍE LITORANEA,2024.0,2413
6,SERRA DA CAPIVARA,2024.0,2881
7,TABULEIROS DO ALTO PARNAÍBA,2024.0,1402
8,VALE DO CANINDÉ,2024.0,550
9,VALE DO RIO GUARIBAS,2024.0,3258


,Territorio Desenvolv,"% de matrículas de tempo integral no ensino regular, na rede estadual",Ano
0,CARAUBAIS,0.139093,2024
1,CHAPADA DAS MANGABEIRAS,0.261027,2024
2,CHAPADA VALE DO ITAIM,0.290949,2024
3,COCAIS,0.186262,2024
4,ENTRE RIOS,0.209044,2024
5,PLANICÍE LITORANEA,0.154759,2024
6,SERRA DA CAPIVARA,0.195694,2024
7,TABULEIROS DO ALTO PARNAÍBA,0.222540,2024
8,VALE DO CANINDÉ,0.172684,2024
9,VALE DO RIO GUARIBAS,0.462193,2024


63 - Indicador - % de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT)

In [21]:
# Quantidade de escolas de tempo integral por estado
tabela_integral_EPT = tabela_Tempo_Integral[["Id Escola", "Territorio Desenvolv", "Ano", "STATUS_INTEGRAL"]]

filt_status_TI = tabela_integral_EPT[tabela_integral_EPT['STATUS_INTEGRAL'] == 1]

# FIltrar o Dataframe onde Matricula_2024 e 2023 é igual a 1
filt_25_TI = filt_status_TI[filt_status_TI['Ano'] == 2025]
filt_24_TI = filt_status_TI[filt_status_TI['Ano'] == 2024]
filt_23_TI = filt_status_TI[filt_status_TI['Ano'] == 2023]
filt_22_TI = filt_status_TI[filt_status_TI['Ano'] == 2022]

base_25_TI = filt_25_TI[["Id Escola", "Territorio Desenvolv"]]
base_24_TI = filt_24_TI[["Id Escola", "Territorio Desenvolv"]]
base_23_TI = filt_23_TI[["Id Escola", "Territorio Desenvolv"]]
base_22_TI = filt_22_TI[["Id Escola", "Territorio Desenvolv"]]

# Remover duplicatas do DataFrame, mantendo apenas a primeira ocorrência
base_25_TI = base_25_TI.drop_duplicates(keep='first')
base_24_TI = base_24_TI.drop_duplicates(keep='first')
base_23_TI = base_23_TI.drop_duplicates(keep='first')
base_22_TI = base_22_TI.drop_duplicates(keep='first')

base_23_TI = pd.concat([base_23_TI, base_22_TI])
base_23_TI['Ano'] = 2023
base_24_TI = pd.concat([base_24_TI, base_23_TI])
base_24_TI['Ano'] = 2024

# Concatenar os DataFrames
df_combinado_g = pd.concat([base_24_TI, base_23_TI], ignore_index=True)
"----------------------------------------------------------------------------------------------------------------"

# Somar os valores na coluna Status_integral por matricula (Id Matricula)
base_25_TI = base_25_TI.groupby('Territorio Desenvolv')['Id Escola'].nunique().reset_index()
base_25_TI['Ano'] = 2025
base_24_TI = base_24_TI.groupby('Territorio Desenvolv')['Id Escola'].nunique().reset_index()
base_24_TI['Ano'] = 2024
base_23_TI = base_23_TI.groupby('Territorio Desenvolv')['Id Escola'].nunique().reset_index()
base_23_TI['Ano'] = 2023
base_22_TI = base_22_TI.groupby('Territorio Desenvolv')['Id Escola'].nunique().reset_index()
base_22_TI['Ano'] = 2022

# Concatenar os DataFrames
df_combinado = pd.concat([base_24_TI, base_23_TI], ignore_index=True)
df_combinado.rename(columns={'Id Escola': 'Qtde de escolas de tempo integral'}, inplace=True)

# Exibir o DataFrame combinado
display(df_combinado)

,Territorio Desenvolv,Qtde de escolas de tempo integral,Ano
0,CARAUBAIS,20,2024
1,CHAPADA DAS MANGABEIRAS,17,2024
2,CHAPADA VALE DO ITAIM,28,2024
3,COCAIS,40,2024
4,ENTRE RIOS,114,2024
5,PLANICÍE LITORANEA,21,2024
6,SERRA DA CAPIVARA,27,2024
7,TABULEIROS DO ALTO PARNAÍBA,17,2024
8,VALE DO CANINDÉ,7,2024
9,VALE DO RIO GUARIBAS,18,2024


In [22]:
# Tabela filtrada
tabela_clientes_TI_EPT = tabela_clientes[["Id Escola", "Territorio Desenvolv", "Matricula_2023", "Matricula_2024", "EtapaAgregada", "ordem"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
filtered_TI_EPT_24 = tabela_clientes_TI_EPT[(tabela_clientes_TI_EPT['Matricula_2024'] == 1) & (tabela_clientes_TI_EPT['ordem'].isin([6, 7, 8, 13]))]
filtered_TI_EPT_24 = filtered_TI_EPT_24[["Id Escola", "Territorio Desenvolv"]]

filtered_TI_EPT_24 = filtered_TI_EPT_24.drop_duplicates(keep='first')

filtered_TI_EPT_23 = tabela_clientes_TI_EPT[(tabela_clientes_TI_EPT['Matricula_2023'] == 1) & (tabela_clientes_TI_EPT['ordem'].isin([6, 7, 8, 13]))]
filtered_TI_EPT_23 = filtered_TI_EPT_23[["Id Escola", "Territorio Desenvolv"]]

filtered_TI_EPT_23 = filtered_TI_EPT_23.drop_duplicates(keep='first')

# Concatenar os DataFrames
df_c_EPT = pd.concat([filtered_TI_EPT_24, filtered_TI_EPT_23], ignore_index=True)
#display(df_c_EPT)
"----------------------------------------------------------------------------------------------------------------"
# Relação inner join para DF final
TI_EPT = df_combinado_g.merge(df_c_EPT, how='inner')
TI_EPT = TI_EPT.drop_duplicates(keep='first')
#display(TI_EPT)
"----------------------------------------------------------------------------------------------------------------"
TI_EPT_24 = TI_EPT[TI_EPT['Ano'] == 2024]
#TI_EPT_24 = TI_EPT_24[["Territorio Desenvolv", "Id Escola", "Ano"]]
TI_EPT_23 = TI_EPT[TI_EPT['Ano'] == 2023]

TI_EPT_24 = TI_EPT_24.groupby(['Territorio Desenvolv', 'Ano'])['Id Escola'].nunique().reset_index()
TI_EPT_23 = TI_EPT_23.groupby(['Territorio Desenvolv', 'Ano'])['Id Escola'].nunique().reset_index()

TI_TI_EPT = pd.concat([TI_EPT_24, TI_EPT_23])
TI_TI_EPT.rename(columns={'Id Escola': 'Qtde de escolas EPT em TI'}, inplace=True)

#display(TI_TI_EPT)

result_TI_EPT = TI_TI_EPT.merge(df_combinado, how='inner')
display(result_TI_EPT)
"----------------------------------------------------------------------------------------------------------------"
# Criar uma nova coluna com a divisão das colunas 'Nº Matriculas Total (EPT)' e 'Nº total de matrículas da rede estadual de educação'
result_TI_EPT['% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT)'] = result_TI_EPT['Qtde de escolas EPT em TI'] / result_TI_EPT['Qtde de escolas de tempo integral']
result_TI_EPT = result_TI_EPT[["Territorio Desenvolv", "% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT)", "Ano"]]
display(result_TI_EPT)

,Territorio Desenvolv,Ano,Qtde de escolas EPT em TI,Qtde de escolas de tempo integral
0,CARAUBAIS,2024,20,20
1,CHAPADA DAS MANGABEIRAS,2024,17,17
2,CHAPADA VALE DO ITAIM,2024,28,28
3,COCAIS,2024,40,40
4,ENTRE RIOS,2024,114,114
5,PLANICÍE LITORANEA,2024,21,21
6,SERRA DA CAPIVARA,2024,27,27
7,TABULEIROS DO ALTO PARNAÍBA,2024,17,17
8,VALE DO CANINDÉ,2024,7,7
9,VALE DO RIO GUARIBAS,2024,18,18


,Territorio Desenvolv,% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT),Ano
0,CARAUBAIS,1.0,2024
1,CHAPADA DAS MANGABEIRAS,1.0,2024
2,CHAPADA VALE DO ITAIM,1.0,2024
3,COCAIS,1.0,2024
4,ENTRE RIOS,1.0,2024
5,PLANICÍE LITORANEA,1.0,2024
6,SERRA DA CAPIVARA,1.0,2024
7,TABULEIROS DO ALTO PARNAÍBA,1.0,2024
8,VALE DO CANINDÉ,1.0,2024
9,VALE DO RIO GUARIBAS,1.0,2024


2213 - Indicador - Nº de matrículas EPT - ok - A Validar

In [23]:
# Tabela filtrada
tabela_clientes_m_ept = tabela_clientes[["idMatricula", "Territorio Desenvolv", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "ordem"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
filtered_df_EPTT_25 = tabela_clientes_m_ept[(tabela_clientes_m_ept['Matricula_2025'] == 1) & (tabela_clientes_m_ept['ordem'].isin([6, 7, 8, 13]))]
filtered_df_EPTT_25 = filtered_df_EPTT_25[["idMatricula", "Territorio Desenvolv", "Ano"]]
filtered_df_EPTT_24 = tabela_clientes_m_ept[(tabela_clientes_m_ept['Matricula_2024'] == 1) & (tabela_clientes_m_ept['ordem'].isin([6, 7, 8, 13]))]
filtered_df_EPTT_24 = filtered_df_EPTT_24[["idMatricula", "Territorio Desenvolv", "Ano"]]
filtered_df_EPTT_23 = tabela_clientes_m_ept[(tabela_clientes_m_ept['Matricula_2023'] == 1) & (tabela_clientes_m_ept['ordem'].isin([6, 7, 8, 13]))]
filtered_df_EPTT_23 = filtered_df_EPTT_23[["idMatricula", "Territorio Desenvolv", "Ano"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "Profissional"
distinct_count_n_EPT_25 = filtered_df_EPTT_25.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()
distinct_count_n_EPT_24 = filtered_df_EPTT_24.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()
distinct_count_n_EPT_23 = filtered_df_EPTT_23.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames de contagem profissional por escola
combined_df_profissional = pd.concat([distinct_count_n_EPT_25, distinct_count_n_EPT_24, distinct_count_n_EPT_23])
combined_df_profissional.rename(columns={'idMatricula': 'Nº de matrículas EPT'}, inplace=True)
combined_df_profissional['Ano'] = combined_df_profissional['Ano'].astype(int)
display(combined_df_profissional)

,Territorio Desenvolv,Ano,Nº de matrículas EPT
0,CARAUBAIS,2024,5260
1,CHAPADA DAS MANGABEIRAS,2024,4026
2,CHAPADA VALE DO ITAIM,2024,5259
3,COCAIS,2024,10123
4,ENTRE RIOS,2024,29098
5,PLANICÍE LITORANEA,2024,6372
6,SERRA DA CAPIVARA,2024,6705
7,TABULEIROS DO ALTO PARNAÍBA,2024,3466
8,VALE DO CANINDÉ,2024,1597
9,VALE DO RIO GUARIBAS,2024,3142


72 - Indicador - Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC) - ok - A Validar

In [24]:
# Tabela filtrada
tabela_clientes_prof = tabela_clientes[["idMatricula", "Territorio Desenvolv", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "EtapaAgregada", "NomeCurso", "NomeEtapa"]]

# Lista de textos a serem contados
cursos_filtrar = [
    "ADMINISTRAÇÃO",
    "CURSO TÉCNICO EM ARTESANATO COM ÊNFASE EM EMPREENDEDORISMO",
    "CONTROLE AMBIENTAL",
    "CURSO TÉCNICO EM DESENVOLVIMENTO DE SISTEMAS",
    "CURSO TÉCNICO EM ELETROTÉCNICA COM ÊNFASE EM ELETROMECÂNICA",
    "CURSO TÉCNICO EM ELETROTÉCNICA COM ÊNFASE EM HIDROGÊNIO VERDE",
    "CURSO TÉCNICO EM GUIA DE TURISMO",
    "CURSO TÉCNICO EM MARKETING DIGITAL",
    "CURSO TÉCNICO EM MINERAÇÃO",
    "PRODUÇÃO DE ÁUDIO E VÍDEO",
    "CURSO TÉCNICO EM PROGRAMAÇÃO DE JOGOS DIGITAIS",
    "CURSO TÉCNICO EM SISTEMAS DE ENERGIA RENOVÁVEL"
]

# Criar uma expressão regular que combine qualquer um dos textos
pattern = '|'.join(cursos_filtrar)

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
df_SEDUCTEC_25 = tabela_clientes_prof[(tabela_clientes_prof['Matricula_2025'] == 1) & (tabela_clientes_prof['NomeCurso'].str.contains(pattern))]
df_SEDUCTEC_25 = df_SEDUCTEC_25[["idMatricula", "Territorio Desenvolv", "Ano", "NomeCurso", "NomeEtapa"]]
df_SEDUCTEC_24 = tabela_clientes_prof[(tabela_clientes_prof['Matricula_2024'] == 1) & (tabela_clientes_prof['NomeCurso'].str.contains(pattern))]
df_SEDUCTEC_24 = df_SEDUCTEC_24[["idMatricula", "Territorio Desenvolv", "Ano", "NomeCurso", "NomeEtapa"]]
df_SEDUCTEC_23 = tabela_clientes_prof[(tabela_clientes_prof['Matricula_2023'] == 1) & (tabela_clientes_prof['NomeCurso'].str.contains(pattern))]
df_SEDUCTEC_23 = df_SEDUCTEC_23[["idMatricula", "Territorio Desenvolv", "Ano", "NomeCurso", "NomeEtapa"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "Profissional"
count_SEDUCTEC_25 = df_SEDUCTEC_25.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()
count_SEDUCTEC_24 = df_SEDUCTEC_24.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()
count_SEDUCTEC_23 = df_SEDUCTEC_23.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames de contagem profissional por escola
combined_df_SEDUCTEC = pd.concat([count_SEDUCTEC_25, count_SEDUCTEC_24, count_SEDUCTEC_23])
combined_df_SEDUCTEC['Ano'] = combined_df_SEDUCTEC['Ano'].astype(int)
combined_df_SEDUCTEC.rename(columns={'idMatricula': 'Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)'}, inplace=True)
display(combined_df_SEDUCTEC)

,Territorio Desenvolv,Ano,Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)
0,CARAUBAIS,2024,2868
1,CHAPADA DAS MANGABEIRAS,2024,1509
2,CHAPADA VALE DO ITAIM,2024,2194
3,COCAIS,2024,4975
4,ENTRE RIOS,2024,12992
5,PLANICÍE LITORANEA,2024,2985
6,SERRA DA CAPIVARA,2024,3164
7,TABULEIROS DO ALTO PARNAÍBA,2024,1470
8,VALE DO CANINDÉ,2024,765
9,VALE DO RIO GUARIBAS,2024,1575


67 - Indicador - Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais - ok - A validar

In [25]:
# Tabela filtrada
tabela_clientes_EJA = tabela_clientes[["idMatricula", "Territorio Desenvolv", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "NomeEtapa"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "EJA"
filtered_df_EJAT_25 = tabela_clientes_EJA[(tabela_clientes_EJA['Matricula_2025'] == 1) & (tabela_clientes_EJA['NomeEtapa'].str.contains("EJA"))]
filtered_df_EJAT_25 = filtered_df_EJAT_25[["idMatricula", "Territorio Desenvolv", "Ano"]]
filtered_df_EJAT_24 = tabela_clientes_EJA[(tabela_clientes_EJA['Matricula_2024'] == 1) & (tabela_clientes_EJA['NomeEtapa'].str.contains("EJA"))]
filtered_df_EJAT_24 = filtered_df_EJAT_24[["idMatricula", "Territorio Desenvolv", "Ano"]]
filtered_df_EJAT_23 = tabela_clientes_EJA[(tabela_clientes_EJA['Matricula_2023'] == 1) & (tabela_clientes_EJA['NomeEtapa'].str.contains("EJA"))]
filtered_df_EJAT_23 = filtered_df_EJAT_23[["idMatricula", "Territorio Desenvolv", "Ano"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA"
EJAT_25 = filtered_df_EJAT_25.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()
EJAT_24 = filtered_df_EJAT_24.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()
EJAT_23 = filtered_df_EJAT_23.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames de contagem EJA por escola
combined_df_EJA = pd.concat([EJAT_25, EJAT_24, EJAT_23])
combined_df_EJA.rename(columns={'idMatricula': 'Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais'}, inplace=True)
combined_df_EJA['Ano'] = combined_df_EJA['Ano'].astype(int)
display(combined_df_EJA)

,Territorio Desenvolv,Ano,Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais
0,CARAUBAIS,2024,3080
1,CHAPADA DAS MANGABEIRAS,2024,1989
2,CHAPADA VALE DO ITAIM,2024,2856
3,COCAIS,2024,4406
4,ENTRE RIOS,2024,16434
5,PLANICÍE LITORANEA,2024,2628
6,SERRA DA CAPIVARA,2024,4694
7,TABULEIROS DO ALTO PARNAÍBA,2024,1525
8,VALE DO CANINDÉ,2024,559
9,VALE DO RIO GUARIBAS,2024,2019


2192 - Indicador - Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT) - ok - A Validar

In [26]:
# Tabela filtrada
tabela_clientes_EJA_EPT = tabela_clientes[["Territorio Desenvolv", "Id Escola", "Ano", "ativa", "ordem", "Matricula_2023", "Matricula_2024", "Matricula_2025", "NomeEtapa"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "EJA"
filtered_m_EJA_EPT_25 = tabela_clientes_EJA_EPT[(tabela_clientes_EJA_EPT['Matricula_2025'] == 1) & (tabela_clientes_EJA_EPT['NomeEtapa'].str.contains("EJA")) & (tabela_clientes_EJA_EPT['ativa'] == 1) & (tabela_clientes_EJA_EPT['ordem'].isin([6, 7, 8, 13]))]
filtered_m_EJA_EPT_25 = filtered_m_EJA_EPT_25[["Id Escola", "Territorio Desenvolv", "Ano"]]
filtered_m_EJA_EPT_24 = tabela_clientes_EJA_EPT[(tabela_clientes_EJA_EPT['Matricula_2024'] == 1) & (tabela_clientes_EJA_EPT['NomeEtapa'].str.contains("EJA")) & (tabela_clientes_EJA_EPT['ativa'] == 1) & (tabela_clientes_EJA_EPT['ordem'].isin([6, 7, 8, 13]))]
filtered_m_EJA_EPT_24 = filtered_m_EJA_EPT_24[["Id Escola", "Territorio Desenvolv", "Ano"]]
filtered_m_EJA_EPT_23 = tabela_clientes_EJA_EPT[(tabela_clientes_EJA_EPT['Matricula_2023'] == 1) & (tabela_clientes_EJA_EPT['NomeEtapa'].str.contains("EJA")) & (tabela_clientes_EJA_EPT['ativa'] == 1) & (tabela_clientes_EJA_EPT['ordem'].isin([6, 7, 8, 13]))]
filtered_m_EJA_EPT_23 = filtered_m_EJA_EPT_23[["Id Escola", "Territorio Desenvolv", "Ano"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA" 'matriculaAtiva'
m_EJA_EPT_25 = filtered_m_EJA_EPT_25.groupby(['Territorio Desenvolv', 'Ano'])['Id Escola'].nunique().reset_index()
m_EJA_EPT_24 = filtered_m_EJA_EPT_24.groupby(['Territorio Desenvolv', 'Ano'])['Id Escola'].nunique().reset_index()
m_EJA_EPT_23 = filtered_m_EJA_EPT_23.groupby(['Territorio Desenvolv', 'Ano'])['Id Escola'].nunique().reset_index()

# Concatenar os DataFrames de contagem por escola
EJA_EPT = pd.concat([m_EJA_EPT_25, m_EJA_EPT_24, m_EJA_EPT_23])
EJA_EPT.rename(columns={'Id Escola': 'Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'}, inplace=True)
EJA_EPT['Ano'] = EJA_EPT['Ano'].astype(int)
display(EJA_EPT)

,Territorio Desenvolv,Ano,Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)
0,CARAUBAIS,2024,22
1,CHAPADA DAS MANGABEIRAS,2024,21
2,CHAPADA VALE DO ITAIM,2024,29
3,COCAIS,2024,31
4,ENTRE RIOS,2024,106
5,PLANICÍE LITORANEA,2024,23
6,SERRA DA CAPIVARA,2024,32
7,TABULEIROS DO ALTO PARNAÍBA,2024,18
8,VALE DO CANINDÉ,2024,7
9,VALE DO RIO GUARIBAS,2024,18


65 - Indicador - Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)

In [27]:
# Tabela filtrada
tabela_clientes_EPT = tabela_clientes[["Territorio Desenvolv", "Id Escola", "Ano", "ativa", "ordem", "Matricula_2023", "Matricula_2024", "Matricula_2025"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "Profissional"
filt_df_EPT_25 = tabela_clientes_EPT[(tabela_clientes_EPT['Matricula_2025'] == 1) & (tabela_clientes_EPT['ativa'] == 1) & (tabela_clientes_EPT['ordem'].isin([6, 7, 8, 13]))]
filt_df_EPT_25 = filt_df_EPT_25[["Territorio Desenvolv", "Ano", "Id Escola"]]
filt_df_EPT_24 = tabela_clientes_EPT[(tabela_clientes_EPT['Matricula_2024'] == 1) & (tabela_clientes_EPT['ativa'] == 1) & (tabela_clientes_EPT['ordem'].isin([6, 7, 8, 13]))]
filt_df_EPT_24 = filt_df_EPT_24[["Territorio Desenvolv", "Ano", "Id Escola"]]
filt_df_EPT_23 = tabela_clientes_EPT[(tabela_clientes_EPT['Matricula_2023'] == 1) & (tabela_clientes_EPT['ativa'] == 1) & (tabela_clientes_EPT['ordem'].isin([6, 7, 8, 13]))]
filt_df_EPT_23 = filt_df_EPT_23[["Territorio Desenvolv", "Ano", "Id Escola"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA"  'matriculaAtiva'
filt_df_EPT_25 = filt_df_EPT_25.groupby(['Territorio Desenvolv', 'Ano'])['Id Escola'].nunique().reset_index()
filt_df_EPT_24 = filt_df_EPT_24.groupby(['Territorio Desenvolv', 'Ano'])['Id Escola'].nunique().reset_index()
filt_df_EPT_23 = filt_df_EPT_23.groupby(['Territorio Desenvolv', 'Ano'])['Id Escola'].nunique().reset_index()

EPT = pd.concat([filt_df_EPT_25, filt_df_EPT_24, filt_df_EPT_23])

# Renomear a coluna de contagem para diferenciar
EPT.rename(columns={'Id Escola': 'Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'}, inplace=True)
EPT['Ano'] = EPT['Ano'].astype(int)
display(EPT)

,Territorio Desenvolv,Ano,Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)
0,CARAUBAIS,2024,31
1,CHAPADA DAS MANGABEIRAS,2024,28
2,CHAPADA VALE DO ITAIM,2024,47
3,COCAIS,2024,61
4,ENTRE RIOS,2024,194
5,PLANICÍE LITORANEA,2024,34
6,SERRA DA CAPIVARA,2024,52
7,TABULEIROS DO ALTO PARNAÍBA,2024,31
8,VALE DO CANINDÉ,2024,9
9,VALE DO RIO GUARIBAS,2024,22


1191 - Indicador - % de escolas da zona rural com oferta de EPT

In [28]:
# Tabela filtrada
tab_rural_EPT_tot = tabela_clientes[["Territorio Desenvolv", "Id Escola", "Ano", "ativa", "ordem", "localizacao", "Matricula_2023", "Matricula_2024", "Matricula_2025"]]
tab_rural_EPT_tot = tab_rural_EPT_tot[(tab_rural_EPT_tot['ativa'] == 1) & (tab_rural_EPT_tot['localizacao'].str.contains("Rural"))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "Profissional"
filt_rural_EPTT_25 = tab_rural_EPT_tot[(tab_rural_EPT_tot['Matricula_2025'] == 1)]
filt_rural_EPTT_25 = filt_rural_EPTT_25[["Territorio Desenvolv", "Id Escola", "Ano"]]
filt_rural_EPTT_24 = tab_rural_EPT_tot[(tab_rural_EPT_tot['Matricula_2024'] == 1)]
filt_rural_EPTT_24 = filt_rural_EPTT_24[["Territorio Desenvolv", "Id Escola", "Ano"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA"  'matriculaAtiva'
filt_rural_EPTT_25 = filt_rural_EPTT_25.groupby(['Territorio Desenvolv', 'Ano'])['Id Escola'].nunique().reset_index()
filt_rural_EPTT_24 = filt_rural_EPTT_24.groupby(['Territorio Desenvolv', 'Ano'])['Id Escola'].nunique().reset_index()

rural_EPTT = pd.concat([filt_rural_EPTT_25, filt_rural_EPTT_24])

# Remover duplicatas com base nas colunas 'Territorio Desenvolv' e 'Ano'
rural_EPTT = rural_EPTT.drop_duplicates(subset=['Territorio Desenvolv', 'Ano'])

# Renomear a coluna de contagem para diferenciar
rural_EPTT.rename(columns={'Id Escola': 'Nº de escolas da zona rural'}, inplace=True)
rural_EPTT['Ano'] = rural_EPTT['Ano'].astype(int)
display(rural_EPTT)

,Territorio Desenvolv,Ano,Nº de escolas da zona rural
0,CARAUBAIS,2024,6
1,CHAPADA DAS MANGABEIRAS,2024,3
2,CHAPADA VALE DO ITAIM,2024,4
3,COCAIS,2024,11
4,ENTRE RIOS,2024,20
5,PLANICÍE LITORANEA,2024,6
6,SERRA DA CAPIVARA,2024,2
7,TABULEIROS DO ALTO PARNAÍBA,2024,5
8,VALE DO RIO GUARIBAS,2024,1
9,VALE DO SAMBITO,2024,2


In [29]:
# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "Profissional"
filt_rural_EPT_25 = tab_rural_EPT_tot[(tab_rural_EPT_tot['Matricula_2025'] == 1) & (tab_rural_EPT_tot['ordem'].isin([6, 7, 8, 13]))]
filt_rural_EPT_25 = filt_rural_EPT_25[["Territorio Desenvolv", "Id Escola", "Ano"]]
filt_rural_EPT_24 = tab_rural_EPT_tot[(tab_rural_EPT_tot['Matricula_2024'] == 1) & (tab_rural_EPT_tot['ordem'].isin([6, 7, 8, 13]))]
filt_rural_EPT_24 = filt_rural_EPT_24[["Territorio Desenvolv", "Id Escola", "Ano"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA"  'matriculaAtiva'
filt_rural_EPT_25 = filt_rural_EPT_25.groupby(['Territorio Desenvolv', 'Ano'])['Id Escola'].nunique().reset_index()
filt_rural_EPT_24 = filt_rural_EPT_24.groupby(['Territorio Desenvolv', 'Ano'])['Id Escola'].nunique().reset_index()

rural_EPT = pd.concat([filt_rural_EPT_24, filt_rural_EPT_25])

# Remover duplicatas com base nas colunas 'Territorio Desenvolv' e 'Ano'
rural_EPT = rural_EPT.drop_duplicates(subset=['Territorio Desenvolv', 'Ano'])

# Renomear a coluna de contagem para diferenciar
rural_EPT.rename(columns={'Id Escola': 'Nº de escolas da zona rural com oferta de EPT'}, inplace=True)
rural_EPT['Ano'] = rural_EPT['Ano'].astype(int)

rural_final = rural_EPTT.merge(rural_EPT, on=['Territorio Desenvolv', 'Ano'], how='left')

rural_final['% de escolas da zona rural com oferta de EPT'] = rural_final['Nº de escolas da zona rural com oferta de EPT'] / rural_final['Nº de escolas da zona rural']
rural_final = rural_final[["Territorio Desenvolv", "% de escolas da zona rural com oferta de EPT", "Ano"]]
display(rural_final)

,Territorio Desenvolv,% de escolas da zona rural com oferta de EPT,Ano
0,CARAUBAIS,0.666667,2024
1,CHAPADA DAS MANGABEIRAS,0.666667,2024
2,CHAPADA VALE DO ITAIM,1.000000,2024
3,COCAIS,0.545455,2024
4,ENTRE RIOS,0.800000,2024
5,PLANICÍE LITORANEA,1.000000,2024
6,SERRA DA CAPIVARA,1.000000,2024
7,TABULEIROS DO ALTO PARNAÍBA,1.000000,2024
8,VALE DO RIO GUARIBAS,1.000000,2024
9,VALE DO SAMBITO,0.500000,2024


2275 - Indicador - % de escolas da zona rural com oferta de Tempo Integral

In [30]:
# Quantidade de escolas da zona rural com oferta de tempo integral por estado
tabela_integral_Ru = tabela_Tempo_Integral[["Id Escola", "Territorio Desenvolv", "localizacao", "ativa", "Ano", "STATUS_INTEGRAL"]]
tabela_integral_Ru = tabela_integral_Ru[(tabela_integral_Ru['ativa'] == 1) & (tabela_integral_Ru['localizacao'].str.contains("Rural"))]

filt_status_Ru = tabela_integral_Ru[tabela_integral_Ru['STATUS_INTEGRAL'] == 1]

# FIltrar o Dataframe onde Matricula_2024 e 2023 é igual a 1
filt_25_Ru = filt_status_Ru[filt_status_Ru['Ano'] == 2025]
filt_24_Ru = filt_status_Ru[filt_status_Ru['Ano'] == 2024]
filt_23_Ru = filt_status_Ru[filt_status_Ru['Ano'] == 2023]
filt_22_Ru = filt_status_Ru[filt_status_Ru['Ano'] == 2022]

base_25_Ru = filt_25_Ru[["Id Escola", "Territorio Desenvolv"]]
base_24_Ru = filt_24_Ru[["Id Escola", "Territorio Desenvolv"]]
base_23_Ru = filt_23_Ru[["Id Escola", "Territorio Desenvolv"]]
base_22_Ru = filt_22_Ru[["Id Escola", "Territorio Desenvolv"]]

# Remover duplicatas do DataFrame, mantendo apenas a primeira ocorrência
base_25_Ru = base_25_Ru.drop_duplicates(keep='first')
base_24_Ru = base_24_Ru.drop_duplicates(keep='first')
base_23_Ru = base_23_Ru.drop_duplicates(keep='first')
base_22_Ru = base_22_Ru.drop_duplicates(keep='first')

base_23_Ru = pd.concat([base_23_Ru, base_22_Ru])
base_23_Ru['Ano'] = 2023
base_24_Ru = pd.concat([base_24_Ru, base_23_Ru])
base_24_Ru['Ano'] = 2024

# Concatenar os DataFrames
df_combinado_Ru = pd.concat([base_24_Ru, base_23_Ru], ignore_index=True)
"----------------------------------------------------------------------------------------------------------------"

# Somar os valores na coluna Status_integral por matricula (Id Matricula)
base_25_Ru = base_25_Ru.groupby('Territorio Desenvolv')['Id Escola'].nunique().reset_index()
base_25_Ru['Ano'] = 2025
base_24_Ru = base_24_Ru.groupby('Territorio Desenvolv')['Id Escola'].nunique().reset_index()
base_24_Ru['Ano'] = 2024
base_23_Ru = base_23_Ru.groupby('Territorio Desenvolv')['Id Escola'].nunique().reset_index()
base_23_Ru['Ano'] = 2023

# Concatenar os DataFrames
df_combinad = pd.concat([base_24_Ru], ignore_index=True)
df_combinad.rename(columns={'Id Escola': 'Qtde de escolas da zona rural com oferta de Tempo Integral'}, inplace=True)

# Exibir o DataFrame combinado
display(df_combinad)

,Territorio Desenvolv,Qtde de escolas da zona rural com oferta de Tempo Integral,Ano
0,CARAUBAIS,1,2024
1,CHAPADA VALE DO ITAIM,2,2024
2,COCAIS,3,2024
3,ENTRE RIOS,4,2024
4,PLANICÍE LITORANEA,2,2024


In [31]:
rural_final_Ru = rural_EPTT.merge(df_combinad, on=['Territorio Desenvolv', 'Ano'], how='left')

rural_final_Ru['% de escolas da zona rural com oferta de Tempo Integral'] = rural_final_Ru['Qtde de escolas da zona rural com oferta de Tempo Integral'] / rural_final_Ru['Nº de escolas da zona rural']
rural_final_Ru = rural_final_Ru[["Territorio Desenvolv", "% de escolas da zona rural com oferta de Tempo Integral", "Ano"]]
display(rural_final_Ru)

,Territorio Desenvolv,% de escolas da zona rural com oferta de Tempo Integral,Ano
0,CARAUBAIS,0.166667,2024
1,CHAPADA DAS MANGABEIRAS,NaN,2024
2,CHAPADA VALE DO ITAIM,0.500000,2024
3,COCAIS,0.272727,2024
4,ENTRE RIOS,0.200000,2024
5,PLANICÍE LITORANEA,0.333333,2024
6,SERRA DA CAPIVARA,NaN,2024
7,TABULEIROS DO ALTO PARNAÍBA,NaN,2024
8,VALE DO RIO GUARIBAS,NaN,2024
9,VALE DO SAMBITO,NaN,2024


2212 - Indicador - Nº de matrículas de EJA integrado ao EPT - ok - A Validar

In [32]:
# Tabela filtrada
tabela_clientes_inte = tabela_clientes[["idMatricula", "Territorio Desenvolv", "Ano", "ativa", "ordem", "Matricula_2023", "Matricula_2024", "Matricula_2025", "NomeEtapa"]]
tabela_clientes_inte = tabela_clientes_inte[(tabela_clientes_inte['NomeEtapa'].str.contains("EJA")) & (tabela_clientes_inte['ativa'] == 1) & (tabela_clientes_inte['ordem'].isin([6, 7, 8, 13]))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "EJA"
filt_df_EJA_EPT_25 = tabela_clientes_inte[(tabela_clientes_inte['Matricula_2025'] == 1)]
filt_df_EJA_EPT_25 = filt_df_EJA_EPT_25[["idMatricula", "Territorio Desenvolv", "Ano"]]
filt_df_EJA_EPT_24 = tabela_clientes_inte[(tabela_clientes_inte['Matricula_2024'] == 1)]
filt_df_EJA_EPT_24 = filt_df_EJA_EPT_24[["idMatricula", "Territorio Desenvolv", "Ano"]]
filt_df_EJA_EPT_23 = tabela_clientes_inte[(tabela_clientes_inte['Matricula_2023'] == 1)]
filt_df_EJA_EPT_23 = filt_df_EJA_EPT_23[["idMatricula", "Territorio Desenvolv", "Ano"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA"  'matriculaAtiva'
dist_EJA_EPT_25 = filt_df_EJA_EPT_25.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()
dist_EJA_EPT_24 = filt_df_EJA_EPT_24.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()
dist_EJA_EPT_23 = filt_df_EJA_EPT_23.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames de contagem por escola
combined_df_EJA_EPT = pd.concat([dist_EJA_EPT_25, dist_EJA_EPT_24, dist_EJA_EPT_23])
combined_df_EJA_EPT.rename(columns={'idMatricula': 'Nº de matrículas de EJA integrado ao EPT'}, inplace=True)
combined_df_EJA_EPT['Ano'] = combined_df_EJA_EPT['Ano'].astype(int)
display(combined_df_EJA_EPT)

,Territorio Desenvolv,Ano,Nº de matrículas de EJA integrado ao EPT
0,CARAUBAIS,2024,1473
1,CHAPADA DAS MANGABEIRAS,2024,1348
2,CHAPADA VALE DO ITAIM,2024,1938
3,COCAIS,2024,3135
4,ENTRE RIOS,2024,12203
5,PLANICÍE LITORANEA,2024,2019
6,SERRA DA CAPIVARA,2024,2925
7,TABULEIROS DO ALTO PARNAÍBA,2024,1197
8,VALE DO CANINDÉ,2024,446
9,VALE DO RIO GUARIBAS,2024,1419


2246 - Indicador - Nº de matrículas do Ensino Médio - ok - A VALIDAR

In [33]:
# Tabela filtrada
tabela_clientes_m = tabela_clientes[["idMatricula", "Territorio Desenvolv", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "nivelEnsino", "EtapaAgregada", "grupo", "STATUS_INTEGRAL"]]

# Filtro 'Ensino Médio'
tabela_clientes_m = tabela_clientes_m[
    (tabela_clientes_m['nivelEnsino'].str.contains('Ensino Médio', regex=False)) &
    (tabela_clientes_m['EtapaAgregada'].str.contains('Ensino Médio|Integrada', regex=True)) &
    (tabela_clientes_m['grupo'].str.contains('Ensino', regex=False))
]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional e Ensino Médio
filt_m_25 = tabela_clientes_m[(tabela_clientes_m['Matricula_2025'] == 1)]
filt_m_25 = filt_m_25[["idMatricula", "Territorio Desenvolv", "Ano", "STATUS_INTEGRAL"]]
filt_m_24 = tabela_clientes_m[(tabela_clientes_m['Matricula_2024'] == 1)]
filt_m_24 = filt_m_24[["idMatricula", "Territorio Desenvolv", "Ano", "STATUS_INTEGRAL"]]

def contar_distintos(df):
    return df.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_m_25, filt_m_24]
combined_df_EM = pd.concat([contar_distintos(df) for df in dataframes])

combined_df_EM.rename(columns={'idMatricula': 'Nº de matrículas do Ensino Médio'}, inplace=True)
combined_df_EM['Ano'] = combined_df_EM['Ano'].astype(int)

display(combined_df_EM)

,Territorio Desenvolv,Ano,Nº de matrículas do Ensino Médio
0,CARAUBAIS,2024,6542
1,CHAPADA DAS MANGABEIRAS,2024,4980
2,CHAPADA VALE DO ITAIM,2024,7201
3,COCAIS,2024,14967
4,ENTRE RIOS,2024,36662
5,PLANICÍE LITORANEA,2024,8565
6,SERRA DA CAPIVARA,2024,7097
7,TABULEIROS DO ALTO PARNAÍBA,2024,3833
8,VALE DO CANINDÉ,2024,1752
9,VALE DO RIO GUARIBAS,2024,3286


61 - Indicador - % de matrículas do ensino médio em tempo integral

In [34]:
# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional e Ensino Médio
filt_m_24 = tabela_clientes_m[(tabela_clientes_m['Matricula_2024'] == 1) & (tabela_clientes_m['STATUS_INTEGRAL'] == 1)]
filt_m_24 = filt_m_24[["idMatricula", "Territorio Desenvolv", "Ano"]]
filt_m_25 = tabela_clientes_m[(tabela_clientes_m['Matricula_2025'] == 1) & (tabela_clientes_m['STATUS_INTEGRAL'] == 1)]
filt_m_25 = filt_m_25[["idMatricula", "Territorio Desenvolv", "Ano"]]

def contar_distintos(df):
    return df.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_m_25, filt_m_24]
combined_TI_EM = pd.concat([contar_distintos(df) for df in dataframes])

combined_TI_EM.rename(columns={'idMatricula': 'Nº de matrículas do Ensino Médio em tempo integral'}, inplace=True)
"--------------------------------------------------------------------------------------------------------------------------------------------------------------------"

result_TI_EM = combined_df_EM.merge(combined_TI_EM, on=['Territorio Desenvolv', 'Ano'], how='left')
result_TI_EM['Ano'] = result_TI_EM['Ano'].astype(int)
"--------------------------------------------------------------------------------------------------------------------------------------------------------------------"
# Criar uma nova coluna com a divisão das colunas 'Nº Matriculas Total (EPT)' e 'Nº total de matrículas da rede estadual de educação'
result_TI_EM['% de matrículas do ensino médio em tempo integral'] = result_TI_EM['Nº de matrículas do Ensino Médio em tempo integral'] / result_TI_EM['Nº de matrículas do Ensino Médio']
result_TI_EM = result_TI_EM[["Territorio Desenvolv", "% de matrículas do ensino médio em tempo integral", "Ano"]]

display(result_TI_EM)

,Territorio Desenvolv,% de matrículas do ensino médio em tempo integral,Ano
0,CARAUBAIS,0.521400,2024
1,CHAPADA DAS MANGABEIRAS,0.380924,2024
2,CHAPADA VALE DO ITAIM,0.428274,2024
3,COCAIS,0.413242,2024
4,ENTRE RIOS,0.452212,2024
5,PLANICÍE LITORANEA,0.423234,2024
6,SERRA DA CAPIVARA,0.372693,2024
7,TABULEIROS DO ALTO PARNAÍBA,0.553613,2024
8,VALE DO CANINDÉ,0.438927,2024
9,VALE DO RIO GUARIBAS,0.466220,2024


2230 - Indicador - Nº de escolas com oferta de tempo integral no ensino médio

In [35]:
# N° De escolas com ensino médio
tabela_clientes_e = tabela_clientes[["Territorio Desenvolv", "Id Escola", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "nivelEnsino", "EtapaAgregada", "grupo", "STATUS_INTEGRAL", "tempoIntegralEscola"]]

# Filtro 'Ensino Médio'
tabela_clientes_e = tabela_clientes_e[
    (tabela_clientes_e['nivelEnsino'].str.contains('Ensino Médio', regex=False)) &
    (tabela_clientes_e['EtapaAgregada'].str.contains('Ensino Médio|Integrada', regex=True)) &
    (tabela_clientes_e['grupo'].str.contains('Ensino', regex=False)) &
    (tabela_clientes_e['tempoIntegralEscola'] == 1)
]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional e Ensino Médio
filtered_e_25 = tabela_clientes_e[(tabela_clientes_e['Matricula_2025'] == 1)]
filtered_e_25 = filtered_e_25[["Id Escola", "Territorio Desenvolv", "Ano"]]
filtered_e_24 = tabela_clientes_e[(tabela_clientes_e['Matricula_2024'] == 1)]
filtered_e_24 = filtered_e_24[["Id Escola", "Territorio Desenvolv", "Ano"]]
filtered_e_23 = tabela_clientes_e[(tabela_clientes_e['Matricula_2023'] == 1)]
filtered_e_23 = filtered_e_23[["Id Escola", "Territorio Desenvolv", "Ano"]]

def contar_distintos(df):
    return df.groupby(['Territorio Desenvolv', 'Ano'])['Id Escola'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filtered_e_25, filtered_e_24, filtered_e_23]
combined_e_EM = pd.concat([contar_distintos(df) for df in dataframes])

combined_e_EM.rename(columns={'Id Escola': 'Nº de escolas com oferta de tempo integral no ensino médio'}, inplace=True)
combined_e_EM['Ano'] = combined_e_EM['Ano'].astype(int)
display(combined_e_EM)

,Territorio Desenvolv,Ano,Nº de escolas com oferta de tempo integral no ensino médio
0,CARAUBAIS,2024,20
1,CHAPADA DAS MANGABEIRAS,2024,17
2,CHAPADA VALE DO ITAIM,2024,28
3,COCAIS,2024,40
4,ENTRE RIOS,2024,115
5,PLANICÍE LITORANEA,2024,21
6,SERRA DA CAPIVARA,2024,27
7,TABULEIROS DO ALTO PARNAÍBA,2024,17
8,VALE DO CANINDÉ,2024,7
9,VALE DO RIO GUARIBAS,2024,18


2208 - Indicador - Nº de cursos EPT ofertados - ok - A Validar

In [36]:
# Tabela filtrada
tabela_clientes_ept_of = tabela_clientes[["idMatricula", "Territorio Desenvolv", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "NomeCurso", "ordem"]]
tabela_clientes_ept_of = tabela_clientes_ept_of[(tabela_clientes_ept_of['ordem'].isin([6, 7, 8, 13]))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
filt_df_EPT_curso_25 = tabela_clientes_ept_of[(tabela_clientes_ept_of['Matricula_2025'] == 1)]
filt_df_EPT_curso_25 = filt_df_EPT_curso_25[["Territorio Desenvolv", "Ano", "NomeCurso"]]
filt_df_EPT_curso_24 = tabela_clientes_ept_of[(tabela_clientes_ept_of['Matricula_2024'] == 1)]
filt_df_EPT_curso_24 = filt_df_EPT_curso_24[["Territorio Desenvolv", "Ano", "NomeCurso"]]
filt_df_EPT_curso_23 = tabela_clientes_ept_of[(tabela_clientes_ept_of['Matricula_2023'] == 1)]
filt_df_EPT_curso_23 = filt_df_EPT_curso_23[["Territorio Desenvolv", "Ano", "NomeCurso"]]

def contar_distintos(df):
    return df.groupby(['Territorio Desenvolv', 'Ano'])['NomeCurso'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_df_EPT_curso_25, filt_df_EPT_curso_24, filt_df_EPT_curso_23]
combined_df_cursos_profissional = pd.concat([contar_distintos(df) for df in dataframes])

combined_df_cursos_profissional.rename(columns={'NomeCurso': 'Nº de cursos EPT ofertados'}, inplace=True)
combined_df_cursos_profissional['Ano'] = combined_df_cursos_profissional['Ano'].astype(int)
display(combined_df_cursos_profissional)

,Territorio Desenvolv,Ano,Nº de cursos EPT ofertados
0,CARAUBAIS,2024,45
1,CHAPADA DAS MANGABEIRAS,2024,57
2,CHAPADA VALE DO ITAIM,2024,50
3,COCAIS,2024,75
4,ENTRE RIOS,2024,112
5,PLANICÍE LITORANEA,2024,73
6,SERRA DA CAPIVARA,2024,56
7,TABULEIROS DO ALTO PARNAÍBA,2024,34
8,VALE DO CANINDÉ,2024,28
9,VALE DO RIO GUARIBAS,2024,35


2232 - Indicador - Nº de matrículas da educação especial (AEE) - ok - A Validar

In [37]:
# Tabela filtrada
tabela_clientes_AEE = tabela_clientes[["idMatricula", "Territorio Desenvolv", "Ano", "NomeCurso", "Matricula_2023", "Matricula_2024", "Matricula_2025"]]
tabela_clientes_AEE = tabela_clientes_AEE[(tabela_clientes_AEE['NomeCurso'] == "AEE")]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
filt_df_AEE_25 = tabela_clientes_AEE[(tabela_clientes_AEE['Matricula_2025'] == 1)]
filt_df_AEE_25 = filt_df_AEE_25[["idMatricula", "Territorio Desenvolv", "Ano"]]
filt_df_AEE_24 = tabela_clientes_AEE[(tabela_clientes_AEE['Matricula_2024'] == 1)]
filt_df_AEE_24 = filt_df_AEE_24[["idMatricula", "Territorio Desenvolv", "Ano"]]
filt_df_AEE_23 = tabela_clientes_AEE[(tabela_clientes_AEE['Matricula_2023'] == 1)]
filt_df_AEE_23 = filt_df_AEE_23[["idMatricula", "Territorio Desenvolv", "Ano"]]

def contar_distintos(df):
    return df.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_df_AEE_25, filt_df_AEE_24, filt_df_AEE_23]
combined_df_AEE = pd.concat([contar_distintos(df) for df in dataframes])

combined_df_AEE.rename(columns={'idMatricula': 'Nº de matrículas da educação especial (AEE)'}, inplace=True)
combined_df_AEE['Ano'] = combined_df_AEE['Ano'].astype(int)
display(combined_df_AEE)

,Territorio Desenvolv,Ano,Nº de matrículas da educação especial (AEE)
0,CARAUBAIS,2024,85
1,CHAPADA DAS MANGABEIRAS,2024,50
2,CHAPADA VALE DO ITAIM,2024,90
3,COCAIS,2024,253
4,ENTRE RIOS,2024,1301
5,PLANICÍE LITORANEA,2024,143
6,SERRA DA CAPIVARA,2024,96
7,TABULEIROS DO ALTO PARNAÍBA,2024,41
8,VALE DO CANINDÉ,2024,36
9,VALE DO RIO GUARIBAS,2024,64


348 - Indicadores - Nº de matrículas transferidas da rede estadual para a rede municipal - ok - A Validar

In [38]:
# Tabela filtrada
tabela_clientes_E_M = tabela_clientes[["idMatricula", "Territorio Desenvolv", "Ano", "NomeEtapa", "Matricula_2023", "Matricula_2024", "Matricula_2025"]]

Etapa_filtrar = [
    "Ensino fundamental de 9 anos - 5º Ano",
    "Ensino fundamental de 9 anos - 6º Ano",
    "Ensino fundamental de 9 anos - 7º Ano",
    "Ensino fundamental de 9 anos - 8º Ano",
    "Ensino fundamental de 9 anos - 9º Ano"
]

# Criar uma expressão regular que combine qualquer um dos textos
pattern = '|'.join(Etapa_filtrar)

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
filt_df_E_M_25 = tabela_clientes_E_M[(tabela_clientes_E_M['Matricula_2025'] == 1) & (tabela_clientes_E_M['NomeEtapa'].str.contains(pattern))]
filt_df_E_M_25 = filt_df_E_M_25[["idMatricula", "Territorio Desenvolv", "Ano"]]
filt_df_E_M_24 = tabela_clientes_E_M[(tabela_clientes_E_M['Matricula_2024'] == 1) & (tabela_clientes_E_M['NomeEtapa'].str.contains(pattern))]
filt_df_E_M_24 = filt_df_E_M_24[["idMatricula", "Territorio Desenvolv", "Ano"]]
filt_df_E_M_23 = tabela_clientes_E_M[(tabela_clientes_E_M['Matricula_2023'] == 1) & (tabela_clientes_E_M['NomeEtapa'].str.contains(pattern))]
filt_df_E_M_23 = filt_df_E_M_23[["idMatricula", "Territorio Desenvolv", "Ano"]]

def contar_distintos(df):
    return df.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_df_E_M_25, filt_df_E_M_24, filt_df_E_M_23]
combined_df_E_M = pd.concat([contar_distintos(df) for df in dataframes])

# Exibir o DataFrame combinado
combined_df_E_M = combined_df_E_M.groupby(['Territorio Desenvolv', 'Ano'])['idMatricula'].sum().reset_index()
combined_df_E_M['Ano'] = combined_df_E_M['Ano'].astype(int)

# Ordena o DataFrame pelo ano (caso não esteja ordenado)
combined_df_E_M = combined_df_E_M.sort_values(by='Ano')

# Calcula a diferença de matrículas ano a ano
combined_df_E_M['Diferença de matrículas'] = combined_df_E_M['idMatricula'].diff()
combined_df_E_M.rename(columns={'Diferença de matrículas': 'Nº de matrículas transferidas da rede estadual para a rede municipal'}, inplace=True)
combined_df_E_M = combined_df_E_M[['Territorio Desenvolv', 'Ano', 'Nº de matrículas transferidas da rede estadual para a rede municipal']]
display(combined_df_E_M)

,Territorio Desenvolv,Ano,Nº de matrículas transferidas da rede estadual para a rede municipal
0,CARAUBAIS,2023,NaN
2,CHAPADA DAS MANGABEIRAS,2023,-48.0
20,VALE DO SAMBITO,2023,205.0
4,CHAPADA VALE DO ITAIM,2023,973.0
6,COCAIS,2023,435.0
18,VALE DO RIO GUARIBAS,2023,-1456.0
8,ENTRE RIOS,2023,9419.0
10,PLANICÍE LITORANEA,2023,-6757.0
22,VALE DOS RIOS PIAUÍ E ITAUEIRA,2023,-3388.0
12,SERRA DA CAPIVARA,2023,1942.0


Mesclagem de DateFrame Parcial

In [39]:
# Mesclar os DataFrames para adicionar as colunas com o 
# 159 - 2213
final_combined_df = combined_df.merge(combined_df_profissional, on=['Territorio Desenvolv', 'Ano'], how='left')
#67
final_combined_df = final_combined_df.merge(combined_df_EJA, on=['Territorio Desenvolv', 'Ano'], how='left')
#2246
final_combined_df = final_combined_df.merge(combined_df_EM, on=['Territorio Desenvolv', 'Ano'], how='left')
#2208
final_combined_df = final_combined_df.merge(combined_df_cursos_profissional, on=['Territorio Desenvolv', 'Ano'], how='left')
#2232
final_combined_df = final_combined_df.merge(combined_df_AEE, on=['Territorio Desenvolv', 'Ano'], how='left')
#72
final_combined_df = final_combined_df.merge(combined_df_SEDUCTEC, on=['Territorio Desenvolv', 'Ano'], how='left')
#2192
final_combined_df = final_combined_df.merge(EJA_EPT, on=['Territorio Desenvolv', 'Ano'], how='left')
#348
final_combined_df = final_combined_df.merge(combined_df_E_M, on=['Territorio Desenvolv', 'Ano'], how='left')
#65
final_combined_df = final_combined_df.merge(EPT, on=['Territorio Desenvolv', 'Ano'], how='left')
#2196
final_combined_df = final_combined_df.merge(matricula_aut_indigena, on=['Territorio Desenvolv', 'Ano'], how='left')
#2215
final_combined_df = final_combined_df.merge(matricula_aut_pretos, on=['Territorio Desenvolv', 'Ano'], how='left')
#2238
final_combined_df = final_combined_df.merge(base_nota_media_M, on=['Territorio Desenvolv', 'Ano'], how='left')
#402
final_combined_df = final_combined_df.merge(base_nota_media_LP, on=['Territorio Desenvolv', 'Ano'], how='left')
#1194
final_combined_df = final_combined_df.merge(nota_LP, on=['Territorio Desenvolv', 'Ano'], how='left')
#1193
final_combined_df = final_combined_df.merge(mat_M, on=['Territorio Desenvolv', 'Ano'], how='left')
#163
final_combined_df = final_combined_df.merge(EM_geral, on=['Territorio Desenvolv', 'Ano'], how='left')
#2237
final_combined_df = final_combined_df.merge(nota_media_LP, on=['Territorio Desenvolv', 'Ano'], how='left')
#381
final_combined_df = final_combined_df.merge(base_nota_media_ITA_IME, on=['Territorio Desenvolv', 'Ano'], how='left')
#382
final_combined_df = final_combined_df.merge(base_nota_media_esp_ITA_IME, on=['Territorio Desenvolv', 'Ano'], how='left')
#385
final_combined_df = final_combined_df.merge(base_nota_freq, on=['Territorio Desenvolv', 'Ano'], how='left')
#2202
final_combined_df = final_combined_df.merge(resultado_final, on=['Territorio Desenvolv', 'Ano'], how='left')
#2212
final_combined_df = final_combined_df.merge(combined_df_EJA_EPT, on=['Territorio Desenvolv', 'Ano'], how='left')
#2230
final_combined_df = final_combined_df.merge(combined_e_EM, on=['Territorio Desenvolv', 'Ano'], how='left')
#63
final_combined_df = final_combined_df.merge(result_TI_EPT, on=['Territorio Desenvolv', 'Ano'], how='left')
#153
final_combined_df = final_combined_df.merge(resultado_mun, on=['Territorio Desenvolv', 'Ano'], how='left')
#61
final_combined_df = final_combined_df.merge(result_TI_EM, on=['Territorio Desenvolv', 'Ano'], how='left')
#85
final_combined_df = final_combined_df.merge(freq_aprovativa, on=['Territorio Desenvolv', 'Ano'], how='left')
#1191
final_combined_df = final_combined_df.merge(rural_final, on=['Territorio Desenvolv', 'Ano'], how='left')
#2275
final_combined_df = final_combined_df.merge(rural_final_Ru, on=['Territorio Desenvolv', 'Ano'], how='left')
"-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------"

# Substituindo os NaN por 0 (ou outro valor apropriado)
#2230
final_combined_df['Nº de escolas com oferta de tempo integral no ensino médio'] = final_combined_df['Nº de escolas com oferta de tempo integral no ensino médio'].fillna(0)
#2212
final_combined_df['Nº de matrículas de EJA integrado ao EPT'] = final_combined_df['Nº de matrículas de EJA integrado ao EPT'].fillna(0)
#2202
final_combined_df['% de matrículas de tempo integral no ensino regular, na rede estadual'] = final_combined_df['% de matrículas de tempo integral no ensino regular, na rede estadual'].fillna(0)
#1194
final_combined_df['% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa'] = final_combined_df['% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa'].fillna(0)
#1193
final_combined_df['% de estudantes do E.M. com média >= 6 no componente curricular Matemática'] = final_combined_df['% de estudantes do E.M. com média >= 6 no componente curricular Matemática'].fillna(0)
#381
final_combined_df['Média nas avaliações nas demais disciplinas'] = final_combined_df['Média nas avaliações nas demais disciplinas'].fillna(0)
#163
final_combined_df['% de aprovação dos alunos do Ensino Médio'] = final_combined_df['% de aprovação dos alunos do Ensino Médio'].fillna(0)
#385
final_combined_df['% frequência dos estudantes inscritos nas turmas ITA/IME'] = final_combined_df['% frequência dos estudantes inscritos nas turmas ITA/IME'].fillna(0)
#382
final_combined_df['Média nas avaliações das disciplinas especificas ITA/IME'] = final_combined_df['Média nas avaliações das disciplinas especificas ITA/IME'].fillna(0)
#402
final_combined_df['Média dos alunos da rede nas disciplinas de línguas estrangeiras'] = final_combined_df['Média dos alunos da rede nas disciplinas de línguas estrangeiras'].fillna(0)
#2237
final_combined_df['Nota média dos estudantes da Rede no componente curricular Língua Portuguesa'] = final_combined_df['Nota média dos estudantes da Rede no componente curricular Língua Portuguesa'].fillna(0)
#2238
final_combined_df['Nota média dos estudantes da Rede no componente curricular Matemática'] = final_combined_df['Nota média dos estudantes da Rede no componente curricular Matemática'].fillna(0)
#2215
final_combined_df['N° de estudantes autodeclarados pretos matriculados na rede'] = final_combined_df['N° de estudantes autodeclarados pretos matriculados na rede'].fillna(0)
#2196
final_combined_df['N° de estudantes autodeclarados indígenas matriculados na rede'] = final_combined_df['N° de estudantes autodeclarados indígenas matriculados na rede'].fillna(0)
#65
final_combined_df['Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'] = final_combined_df['Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'].fillna(0)
#348
final_combined_df['Nº de matrículas transferidas da rede estadual para a rede municipal'] = final_combined_df['Nº de matrículas transferidas da rede estadual para a rede municipal'].fillna(0)
#2192
final_combined_df['Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'] = final_combined_df['Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'].fillna(0)
#72
final_combined_df['Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)'] = final_combined_df['Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)'].fillna(0)
#2232
final_combined_df['Nº de matrículas da educação especial (AEE)'] = final_combined_df['Nº de matrículas da educação especial (AEE)'].fillna(0)
#2213
final_combined_df['Nº de matrículas EPT'] = final_combined_df['Nº de matrículas EPT'].fillna(0)
#67
final_combined_df['Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais'] = final_combined_df['Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais'].fillna(0)
#2246
final_combined_df['Nº de matrículas do Ensino Médio'] = final_combined_df['Nº de matrículas do Ensino Médio'].fillna(0)
#2208
final_combined_df['Nº de cursos EPT ofertados'] = final_combined_df['Nº de cursos EPT ofertados'].fillna(0)
#153
final_combined_df['% de municípios com escolas da rede estadual em tempo integral'] = final_combined_df['% de municípios com escolas da rede estadual em tempo integral'].fillna(0)
#61
final_combined_df['% de matrículas do ensino médio em tempo integral'] = final_combined_df['% de matrículas do ensino médio em tempo integral'].fillna(0)
#1191
final_combined_df['% de escolas da zona rural com oferta de EPT'] = final_combined_df['% de escolas da zona rural com oferta de EPT'].fillna(0)
#85
final_combined_df['% de estudantes da rede estadual com frequência regular em 75%'] = final_combined_df['% de estudantes da rede estadual com frequência regular em 75%'].fillna(0)
#2275
final_combined_df['% de escolas da zona rural com oferta de Tempo Integral'] = final_combined_df['% de escolas da zona rural com oferta de Tempo Integral'].fillna(0)
"-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------"
#2230
final_combined_df['Nº de escolas com oferta de tempo integral no ensino médio'] = final_combined_df['Nº de escolas com oferta de tempo integral no ensino médio'].astype(int)
#2212
final_combined_df['Nº de matrículas de EJA integrado ao EPT'] = final_combined_df['Nº de matrículas de EJA integrado ao EPT'].astype(int)
#2213
final_combined_df['Nº de matrículas EPT'] = final_combined_df['Nº de matrículas EPT'].astype(int)
#67
final_combined_df['Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais'] = final_combined_df['Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais'].astype(int)
#2246
final_combined_df['Nº de matrículas do Ensino Médio'] = final_combined_df['Nº de matrículas do Ensino Médio'].astype(int)
#2208
final_combined_df['Nº de cursos EPT ofertados'] = final_combined_df['Nº de cursos EPT ofertados'].astype(int)
#2232
final_combined_df['Nº de matrículas da educação especial (AEE)'] = final_combined_df['Nº de matrículas da educação especial (AEE)'].astype(float).astype(int)
#72
final_combined_df['Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)'] = final_combined_df['Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)'].astype(float).astype(int)
#2192
final_combined_df['Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'] = final_combined_df['Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'].astype(float).astype(int)
#348
final_combined_df['Nº de matrículas transferidas da rede estadual para a rede municipal'] = final_combined_df['Nº de matrículas transferidas da rede estadual para a rede municipal'].astype(float).astype(int)
#65
final_combined_df['Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'] = final_combined_df['Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'].astype(float).astype(int)
#2196
final_combined_df['N° de estudantes autodeclarados indígenas matriculados na rede'] = final_combined_df['N° de estudantes autodeclarados indígenas matriculados na rede'].astype(float).astype(int)
#2215
final_combined_df['N° de estudantes autodeclarados pretos matriculados na rede'] = final_combined_df['N° de estudantes autodeclarados pretos matriculados na rede'].astype(float).astype(int)
#2238
final_combined_df['Nota média dos estudantes da Rede no componente curricular Matemática'] = final_combined_df['Nota média dos estudantes da Rede no componente curricular Matemática'].astype(float).astype(int)
#2237
final_combined_df['Nota média dos estudantes da Rede no componente curricular Língua Portuguesa'] = final_combined_df['Nota média dos estudantes da Rede no componente curricular Língua Portuguesa'].astype(float).astype(int)
#402
final_combined_df['Média dos alunos da rede nas disciplinas de línguas estrangeiras'] = final_combined_df['Média dos alunos da rede nas disciplinas de línguas estrangeiras'].astype(float).astype(int)
#382
final_combined_df['Média nas avaliações das disciplinas especificas ITA/IME'] = final_combined_df['Média nas avaliações das disciplinas especificas ITA/IME'].astype(float).astype(int)
#381
final_combined_df['Média nas avaliações nas demais disciplinas'] = final_combined_df['Média nas avaliações nas demais disciplinas'].astype(float).astype(int)

final_combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 32 columns):
 #   Column                                                                                      Non-Null Count  Dtype  
---  ------                                                                                      --------------  -----  
 0   Territorio Desenvolv                                                                        24 non-null     object 
 1   Ano                                                                                         24 non-null     int32  
 2   Nº total de matrículas da rede estadual de educação                                         24 non-null     int64  
 3   Nº de matrículas EPT                                                                        24 non-null     int32  
 4   Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais                 24 non-null     int32  
 5   Nº de matrículas do Ensino Médio             

66 - Indicador - % de matrículas de EPT sobre o total de matrículas - ok - A Validar

Mesclagem de DateFrame Parcial

In [40]:
# Realizar o merge entre os dois DataFrames com base nas colunas 'Id Escola' e 'Ano'
final_percent_df = pd.merge(combined_df_profissional, combined_df, on=['Territorio Desenvolv', 'Ano'])

# Criar uma nova coluna com a divisão das colunas 'Nº Matriculas Total (EPT)' e 'Nº total de matrículas da rede estadual de educação'
final_percent_df['% de matrículas de EPT sobre o total de matrículas'] = final_percent_df['Nº de matrículas EPT'] / final_percent_df['Nº total de matrículas da rede estadual de educação']

final_percent_df['% de matrículas de EPT sobre o total de matrículas'] = final_percent_df['% de matrículas de EPT sobre o total de matrículas'].apply(lambda x: f"{x:.4f}").astype(float)

final_percent_df['Territorio Desenvolv'] = final_percent_df['Territorio Desenvolv'].astype(str)

final_percent_df = final_percent_df[["Territorio Desenvolv", "Ano", "% de matrículas de EPT sobre o total de matrículas"]]

# Mesclagem para DF final
final_percent_df = final_combined_df.merge(final_percent_df, on=['Territorio Desenvolv', 'Ano'], how='left')
# Exibir o DataFrame final combinado
print("Resultado final combinado:")
display(final_percent_df)

Resultado final combinado:


,Territorio Desenvolv,Ano,Nº total de matrículas da rede estadual de educação,Nº de matrículas EPT,Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais,Nº de matrículas do Ensino Médio,Nº de cursos EPT ofertados,Nº de matrículas da educação especial (AEE),Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC),Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT),...,"% de matrículas de tempo integral no ensino regular, na rede estadual",Nº de matrículas de EJA integrado ao EPT,Nº de escolas com oferta de tempo integral no ensino médio,% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT),% de municípios com escolas da rede estadual em tempo integral,% de matrículas do ensino médio em tempo integral,% de estudantes da rede estadual com frequência regular em 75%,% de escolas da zona rural com oferta de EPT,% de escolas da zona rural com oferta de Tempo Integral,% de matrículas de EPT sobre o total de matrículas
0,CARAUBAIS,2023,14016,4614,3288,0,40,53,1179,19,...,0.000000,767,20,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3292
1,CHAPADA DAS MANGABEIRAS,2023,9356,3901,1758,0,38,59,609,19,...,0.000000,598,17,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.4170
2,CHAPADA VALE DO ITAIM,2023,14522,4702,2904,0,35,89,914,28,...,0.000000,1108,27,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3238
3,COCAIS,2023,23687,7439,3181,0,60,176,2388,33,...,0.000000,1271,38,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3141
4,ENTRE RIOS,2023,66722,20071,11284,0,79,1149,5535,127,...,0.000000,5410,103,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3008
5,PLANICÍE LITORANEA,2023,15198,4203,2103,0,52,142,1257,24,...,0.000000,955,21,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.2765
6,SERRA DA CAPIVARA,2023,16223,5300,4098,0,38,95,1363,33,...,0.000000,1298,27,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3267
7,TABULEIROS DO ALTO PARNAÍBA,2023,6906,2830,1004,0,34,32,678,14,...,0.000000,323,17,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.4098
8,VALE DO CANINDÉ,2023,3283,1406,493,0,18,35,567,6,...,0.000000,378,7,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.4283
9,VALE DO RIO GUARIBAS,2023,7017,2208,1389,0,27,53,887,16,...,0.000000,559,18,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3147


2266 - Indicador - % de matrículas de EJA integrado ao EPT (rede estadual)

In [41]:
# Criar uma nova coluna com a divisão das colunas 'Nº de matrículas de EJA integrado ao EPT' e 'Nº total de matrículas da rede estadual de educação'
final_percent_df['% de matrículas de EJA integrado ao EPT (rede estadual)'] = final_percent_df['Nº de matrículas de EJA integrado ao EPT'] / final_percent_df['Nº total de matrículas da rede estadual de educação']

display(final_percent_df)

,Territorio Desenvolv,Ano,Nº total de matrículas da rede estadual de educação,Nº de matrículas EPT,Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais,Nº de matrículas do Ensino Médio,Nº de cursos EPT ofertados,Nº de matrículas da educação especial (AEE),Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC),Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT),...,Nº de matrículas de EJA integrado ao EPT,Nº de escolas com oferta de tempo integral no ensino médio,% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT),% de municípios com escolas da rede estadual em tempo integral,% de matrículas do ensino médio em tempo integral,% de estudantes da rede estadual com frequência regular em 75%,% de escolas da zona rural com oferta de EPT,% de escolas da zona rural com oferta de Tempo Integral,% de matrículas de EPT sobre o total de matrículas,% de matrículas de EJA integrado ao EPT (rede estadual)
0,CARAUBAIS,2023,14016,4614,3288,0,40,53,1179,19,...,767,20,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3292,0.054723
1,CHAPADA DAS MANGABEIRAS,2023,9356,3901,1758,0,38,59,609,19,...,598,17,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.4170,0.063916
2,CHAPADA VALE DO ITAIM,2023,14522,4702,2904,0,35,89,914,28,...,1108,27,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3238,0.076298
3,COCAIS,2023,23687,7439,3181,0,60,176,2388,33,...,1271,38,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3141,0.053658
4,ENTRE RIOS,2023,66722,20071,11284,0,79,1149,5535,127,...,5410,103,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3008,0.081083
5,PLANICÍE LITORANEA,2023,15198,4203,2103,0,52,142,1257,24,...,955,21,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.2765,0.062837
6,SERRA DA CAPIVARA,2023,16223,5300,4098,0,38,95,1363,33,...,1298,27,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3267,0.080010
7,TABULEIROS DO ALTO PARNAÍBA,2023,6906,2830,1004,0,34,32,678,14,...,323,17,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.4098,0.046771
8,VALE DO CANINDÉ,2023,3283,1406,493,0,18,35,567,6,...,378,7,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.4283,0.115139
9,VALE DO RIO GUARIBAS,2023,7017,2208,1389,0,27,53,887,16,...,559,18,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3147,0.079664


2207 - Indicadores - % de matrículas EPT no EM sobre o total de matrículas - ok - A Validar 

In [42]:
# Criar uma nova coluna com a divisão das colunas 'Nº de matrículas de EJA integrado ao EPT' e 'Nº total de matrículas da rede estadual de educação'
final_percent_df['% de matrículas EPT no EM sobre o total de matrículas'] = final_percent_df['Nº de matrículas do Ensino Médio'] / final_percent_df['Nº total de matrículas da rede estadual de educação']

final_percent_df['% de matrículas EPT no EM sobre o total de matrículas'] = final_percent_df['% de matrículas de EPT sobre o total de matrículas'].apply(lambda x: f"{x:.4f}").astype(float)

display(final_percent_df)

,Territorio Desenvolv,Ano,Nº total de matrículas da rede estadual de educação,Nº de matrículas EPT,Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais,Nº de matrículas do Ensino Médio,Nº de cursos EPT ofertados,Nº de matrículas da educação especial (AEE),Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC),Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT),...,Nº de escolas com oferta de tempo integral no ensino médio,% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT),% de municípios com escolas da rede estadual em tempo integral,% de matrículas do ensino médio em tempo integral,% de estudantes da rede estadual com frequência regular em 75%,% de escolas da zona rural com oferta de EPT,% de escolas da zona rural com oferta de Tempo Integral,% de matrículas de EPT sobre o total de matrículas,% de matrículas de EJA integrado ao EPT (rede estadual),% de matrículas EPT no EM sobre o total de matrículas
0,CARAUBAIS,2023,14016,4614,3288,0,40,53,1179,19,...,20,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3292,0.054723,0.3292
1,CHAPADA DAS MANGABEIRAS,2023,9356,3901,1758,0,38,59,609,19,...,17,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.4170,0.063916,0.4170
2,CHAPADA VALE DO ITAIM,2023,14522,4702,2904,0,35,89,914,28,...,27,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3238,0.076298,0.3238
3,COCAIS,2023,23687,7439,3181,0,60,176,2388,33,...,38,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3141,0.053658,0.3141
4,ENTRE RIOS,2023,66722,20071,11284,0,79,1149,5535,127,...,103,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3008,0.081083,0.3008
5,PLANICÍE LITORANEA,2023,15198,4203,2103,0,52,142,1257,24,...,21,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.2765,0.062837,0.2765
6,SERRA DA CAPIVARA,2023,16223,5300,4098,0,38,95,1363,33,...,27,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3267,0.080010,0.3267
7,TABULEIROS DO ALTO PARNAÍBA,2023,6906,2830,1004,0,34,32,678,14,...,17,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.4098,0.046771,0.4098
8,VALE DO CANINDÉ,2023,3283,1406,493,0,18,35,567,6,...,7,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.4283,0.115139,0.4283
9,VALE DO RIO GUARIBAS,2023,7017,2208,1389,0,27,53,887,16,...,18,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3147,0.079664,0.3147


In [43]:
final_percent_df['% de matrículas de EPT sobre o total de matrículas'] = final_percent_df['% de matrículas de EPT sobre o total de matrículas'].replace('.', ',')

# Selecionar automaticamente todas as colunas para value_vars, exceto as que estão em id_vars
id_vars = ['Territorio Desenvolv', 'Ano']
value_vars = [col for col in final_percent_df.columns if col not in id_vars]

# Transformar o DataFrame para a forma PIVOT
long_df = final_percent_df.melt(id_vars=id_vars, 
                                 value_vars=value_vars,
                                 var_name='Indicador', value_name='Quantidade')

# Pivotar para obter as colunas separadas para os anos 2023 e 2024
pivot_df = long_df.pivot_table(index=['Territorio Desenvolv', 'Indicador', 'Ano'], values='Quantidade').reset_index()

# Renomear as colunas para torná-las mais legíveis
pivot_df.columns.name = None
pivot_df.rename(columns={2023: '2023', 2024: '2024'}, inplace=True)

# Seleciona as colunas "CasasDecimais" e "Codigo Indicador" do dataframe especificacao_superintendência_df
resumo_calculado = tabela_especificacao_ind_O_E[['Indicador', 'CasasDecimais', 'Codigo Indicador', 'UnidadeMedida', 'TipoIndicadorEstPro']]
# Mesclar os DataFrames para adicionar as colunas com o cálculo
pivot_df = resumo_calculado.merge(pivot_df, on='Indicador', how='left')

pivot_df['CasasDecimais'] = pivot_df['CasasDecimais'].fillna(0)
pivot_df['CasasDecimais'] = pivot_df['CasasDecimais'].astype(int)
pivot_df['Codigo Indicador'] = pivot_df['Codigo Indicador'].fillna(0)
pivot_df['Codigo Indicador'] = pivot_df['Codigo Indicador'].astype(int)
pivot_df['Ano'] = pivot_df['Ano'].fillna(0)
pivot_df['Ano'] = pivot_df['Ano'].astype(int)

pivot_df = pivot_df[['Territorio Desenvolv', 'Codigo Indicador', 'Ano', 'Quantidade', 'TipoIndicadorEstPro']]

## IDENTIFICAR MOTIVO DE DUPLICAÇÃO DE LINHAS DO DATAFRAME
pivot_df = pivot_df.drop_duplicates(keep='first')

from datetime import datetime

# Capturar a data e hora atuais
data_hora_execucao = datetime.now().strftime('%d/%m/%Y %H:%M:%S')

# Extrair o ano atual
ano_atual = datetime.now().year

# Adicionar uma nova coluna ao DataFrame com a data e hora da última execução do script
pivot_df['Data Cadastro'] = data_hora_execucao# Condicional para definir 'Mes_Referente'
# Se o ano for igual ao ano atual, extrai o mês da data atual, senão atribui o valor 12
pivot_df['Mes_Referente'] = pivot_df.apply(
    lambda row: pd.to_datetime(data_hora_execucao, format='%d/%m/%Y %H:%M:%S').month 
                if row['Ano'] == ano_atual 
                else 12, axis=1
)

pivot_df['Mes_Referente'] = pivot_df['Mes_Referente'].astype(int)

pivot_df = pivot_df[['Territorio Desenvolv', 'Codigo Indicador', 'Ano', 'Quantidade', 'Data Cadastro', 'Mes_Referente', 'TipoIndicadorEstPro']]

# Exibindo a tabela final pivotada
#print("Tabela Final Pivotada:")
#display(pivot_df)
#pivot_df.info()

Preparação e definição de Base

In [2]:
# Preencher valores NaN na coluna 'Hierarquia' com string vazia
tabela_base_gerada['Hierarquia'] = tabela_base_gerada['Hierarquia'].fillna('')

# Filtro por escola
BASE_GERADA = tabela_base_gerada[(tabela_base_gerada['Hierarquia'].str.contains("Territorio_Desenvolv"))]

# Adicionar a nova coluna 'TipoIndicadorEstPro' com a informação 'Manual' em todas as linhas

BASE_GERADA = BASE_GERADA[['Territorio Desenvolv', 'Codigo Indicador', 'Ano', 'Quantidade', 'Data Cadastro', 'Mes_Referente', 'TipoIndicadorEstPro']]

BASE_GERADA['Territorio Desenvolv'] = BASE_GERADA['Territorio Desenvolv'].fillna(0)
BASE_GERADA['Territorio Desenvolv'] = BASE_GERADA['Territorio Desenvolv'].astype(str)

# Exibir o DataFrame
display(BASE_GERADA)
BASE_GERADA.info()

,Territorio Desenvolv,Codigo Indicador,Ano,Quantidade,Data Cadastro,Mes_Referente,TipoIndicadorEstPro
2562,ENTRE RIOS,2222,2022,6.00,2024-09-17 09:17:35,12,E
2563,ENTRE RIOS,2222,2023,6.00,2024-09-17 09:17:35,12,E
2564,ENTRE RIOS,2222,2024,6.00,2024-09-17 09:17:35,7,E
2576,PLANICÍE LITORANEA,2251,2022,603.86,2024-09-17 09:42:43,12,None
2577,COCAIS,2251,2022,598.47,2024-09-17 09:42:43,12,None
...,...,...,...,...,...,...,...
15808,TABULEIROS DO ALTO PARNAÍBA,2279,2024,5.00,2024-11-11 14:24:36,11,E
15809,VALE DO CANINDÉ,2279,2024,3.00,2024-11-11 14:24:36,11,E
15810,VALE DO RIO GUARIBAS,2279,2024,10.00,2024-11-11 14:24:36,11,E
15811,VALE DO SAMBITO,2279,2024,11.00,2024-11-11 14:24:36,11,E


<class 'pandas.core.frame.DataFrame'>
Index: 88 entries, 2562 to 15812
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Territorio Desenvolv  88 non-null     object        
 1   Codigo Indicador      88 non-null     object        
 2   Ano                   88 non-null     int64         
 3   Quantidade            88 non-null     float64       
 4   Data Cadastro         88 non-null     datetime64[ns]
 5   Mes_Referente         88 non-null     int64         
 6   TipoIndicadorEstPro   18 non-null     object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(3)
memory usage: 5.5+ KB


In [3]:
# Mesclar os DataFrames para adicionar as colunas com o cálculo
#base_indicadores = pd.concat([BASE_GERADA, pivot_df], ignore_index=True)
base_indicadores = BASE_GERADA
# Remover duplicatas baseadas em todas as colunas, mantendo apenas a última ocorrência
base_indicadores.drop_duplicates(keep='last', inplace=True)
display(base_indicadores)

,Territorio Desenvolv,Codigo Indicador,Ano,Quantidade,Data Cadastro,Mes_Referente,TipoIndicadorEstPro
2562,ENTRE RIOS,2222,2022,6.00,2024-09-17 09:17:35,12,E
2563,ENTRE RIOS,2222,2023,6.00,2024-09-17 09:17:35,12,E
2564,ENTRE RIOS,2222,2024,6.00,2024-09-17 09:17:35,7,E
2576,PLANICÍE LITORANEA,2251,2022,603.86,2024-09-17 09:42:43,12,None
2577,COCAIS,2251,2022,598.47,2024-09-17 09:42:43,12,None
...,...,...,...,...,...,...,...
15808,TABULEIROS DO ALTO PARNAÍBA,2279,2024,5.00,2024-11-11 14:24:36,11,E
15809,VALE DO CANINDÉ,2279,2024,3.00,2024-11-11 14:24:36,11,E
15810,VALE DO RIO GUARIBAS,2279,2024,10.00,2024-11-11 14:24:36,11,E
15811,VALE DO SAMBITO,2279,2024,11.00,2024-11-11 14:24:36,11,E


In [5]:
# O script atualiza diariamente um arquivo CSV acumulando dados e, no primeiro dia de cada mês, cria um backup mensal com os dados do último dia do mês anterior.

from datetime import datetime, timedelta
import os
import shutil

# Verificar se a pasta historico_diario existe, se não, criar
#historico_diario_path = 'historico_diario'
%run Caminhos.ipynb

# Caminhos para as pastas de histórico diário e mensal
historico_diario_path = os.path.join(pasta_hierarquia_historico, 'historico_diario')
historico_mensal_path = os.path.join(pasta_hierarquia_historico, 'historico_mensal')

# Verificar se a pasta historico_diario existe, se não, criar
if not os.path.exists(historico_diario_path):
    os.makedirs(historico_diario_path)

# Nome do arquivo CSV diário
arquivo_diario = os.path.join(historico_diario_path, 'dados_diario_territorio_des.csv')

# Verificar se o arquivo já existe
if os.path.exists(arquivo_diario):
    # Se o arquivo já existir, abrir em modo de apêndice e sem o cabeçalho
    base_indicadores.to_csv(arquivo_diario, mode='a', header=False, index=False)
else:
    # Se o arquivo não existir, criar um novo com o cabeçalho
    base_indicadores.to_csv(arquivo_diario, index=False)

# Verificar se hoje é o primeiro dia do mês
hoje = datetime.now()
if hoje.day == 1:
    # Obter a data do último dia do mês anterior
    ultimo_dia_mes_anterior = hoje.replace(day=1) - timedelta(days=1)
    
    # Verificar se o arquivo diário tem entradas para o último dia do mês anterior
    dados_diarios = pd.read_csv(arquivo_diario)
    dados_ultimo_dia = dados_diarios[dados_diarios['Data Cadastro'].str.startswith(ultimo_dia_mes_anterior.strftime('%Y-%m-%d'))]

    if not dados_ultimo_dia.empty:
        # Verificar se a pasta historico_mensal existe, se não, criar
        if not os.path.exists(historico_mensal_path):
            os.makedirs(historico_mensal_path)
        
        # Salvar os dados do último dia do mês anterior no arquivo mensal
        dados_ultimo_dia.to_csv(os.path.join(historico_mensal_path, f'dados_territorio_des{ultimo_dia_mes_anterior.strftime("%Y-%m-%d")}.csv'), index=False)